In [1]:
# EquiBind Batch Docking Pipeline with Multiple Pose Generation
#
# This notebook docks all protein-ligand combinations using EquiBind,
# generating multiple diverse poses per combination using RDKit conformer generation.
#
# Strategy: Generate multiple 3D conformers with RDKit's EmbedMultipleConfs(),
# then run EquiBind on each conformer to produce diverse docked poses.
#
# Uses conda environment 'equibind' to run multiligand_inference.py:
#   conda run -n equibind python ~/docking_tools/EquiBind/multiligand_inference.py \
#     -o ./equibind_out -r protein.pdb -l ligand.sdf --device cpu

In [2]:
from __future__ import annotations

import hashlib
import itertools
import json
import os
import shutil
import subprocess
import time
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import numpy as np
from collections import defaultdict

# RDKit imports for conformer generation
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign

In [3]:
# ============================================================================
# CONFIGURATION - Adjust these parameters as needed
# ============================================================================

# Paths
workspace_root = Path.cwd()
drugs_dir = workspace_root / "Drugs"
receptors_dir = workspace_root / "Orai"

# EquiBind paths
EQUIBIND_DIR = Path("/home/manndo/docking_tools/EquiBind")
EQUIBIND_MULTILIGAND_SCRIPT = EQUIBIND_DIR / "multiligand_inference.py"

# Number of unique poses to generate per protein-ligand combination
NUM_POSES: int = 30

# Number of RDKit conformers to generate (should be >= NUM_POSES, extras for diversity)
NUM_CONFORMERS: int = 10

# Maximum docking attempts per combination (fallback if conformers fail)
MAX_ATTEMPTS: int = NUM_POSES * 30

# Minimum RMSD (in Angstroms) between poses to consider them distinct
POSE_RMSD_THRESHOLD: float = 1.0

# RDKit conformer generation parameters
RDKIT_RANDOM_SEED: int = 42
RDKIT_NUM_THREADS: int = 0  # 0 = use all available threads
RDKIT_PRUNE_RMS_THRESH: float = 0.5  # Prune similar conformers during generation

# Device for EquiBind (cpu or cuda)
EQUIBIND_DEVICE = "cpu"

# Output directories
EQUIBIND_BATCH_DIR = workspace_root / "equibind_batches"
EQUIBIND_OUTPUT_DIR = workspace_root / "equibind_docked_poses"
EQUIBIND_BATCH_DIR.mkdir(parents=True, exist_ok=True)
EQUIBIND_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Check for reduce executable (for adding hydrogens to proteins)
REDUCE_EXECUTABLE = shutil.which("reduce")

# Overwrite settings
OVERWRITE_EXISTING = False  # Set to True to re-dock existing combinations

# Validate directories
if not drugs_dir.exists():
    raise FileNotFoundError(f"Ligand directory missing: {drugs_dir}")
if not receptors_dir.exists():
    raise FileNotFoundError(f"Receptor directory missing: {receptors_dir}")

print("=" * 80)
print("EquiBind Batch Docking Configuration")
print("=" * 80)
print(f"Number of poses per combination: {NUM_POSES}")
print(f"RDKit conformers to generate: {NUM_CONFORMERS}")
print(f"Maximum attempts per combination: {MAX_ATTEMPTS}")
print(f"Pose RMSD threshold: {POSE_RMSD_THRESHOLD} Å")
print(f"RDKit prune RMS threshold: {RDKIT_PRUNE_RMS_THRESH} Å")
print(f"EquiBind script: {EQUIBIND_MULTILIGAND_SCRIPT}")
print(f"EquiBind device: {EQUIBIND_DEVICE}")
print(f"Batch directory: {EQUIBIND_BATCH_DIR}")
print(f"Output directory: {EQUIBIND_OUTPUT_DIR}")
print(f"Reduce executable: {REDUCE_EXECUTABLE or 'NOT FOUND'}")
print()

# Validate EquiBind installation
if not EQUIBIND_MULTILIGAND_SCRIPT.exists():
    print(f"⚠️  WARNING: EquiBind multiligand_inference.py not found at {EQUIBIND_MULTILIGAND_SCRIPT}")
    print("   Please update EQUIBIND_DIR to point to your EquiBind installation")
else:
    print(f"✓ EquiBind script found")

EquiBind Batch Docking Configuration
Number of poses per combination: 30
RDKit conformers to generate: 10
Maximum attempts per combination: 900
Pose RMSD threshold: 1.0 Å
RDKit prune RMS threshold: 0.5 Å
EquiBind script: /home/manndo/docking_tools/EquiBind/multiligand_inference.py
EquiBind device: cpu
Batch directory: /home/manndo/MasterProject/equibind_batches
Output directory: /home/manndo/MasterProject/equibind_docked_poses
Reduce executable: /home/manndo/anaconda3/envs/equibind/bin/reduce

✓ EquiBind script found


In [4]:
@dataclass
class PoseInfo:
    """Information about a single docked pose."""
    pose_id: int
    sdf_path: Path
    centroid: Tuple[float, float, float]
    seed: int
    conformer_id: int
    
    def to_dict(self) -> dict:
        return {
            "pose_id": self.pose_id,
            "sdf_path": str(self.sdf_path),
            "centroid": list(self.centroid),
            "seed": self.seed,
            "conformer_id": self.conformer_id,
        }


@dataclass
class DockingResult:
    """Result of docking a protein-ligand combination."""
    protein_name: str
    ligand_name: str
    protein_path: Path
    ligand_path: Path
    output_dir: Path
    status: str  # "success", "partial", "failed", "skipped"
    poses: List[PoseInfo] = field(default_factory=list)
    num_attempts: int = 0
    num_conformers_generated: int = 0
    error_message: str = ""
    elapsed_time: float = 0.0
    
    def to_dict(self) -> dict:
        return {
            "protein_name": self.protein_name,
            "ligand_name": self.ligand_name,
            "protein_path": str(self.protein_path),
            "ligand_path": str(self.ligand_path),
            "output_dir": str(self.output_dir),
            "status": self.status,
            "poses": [p.to_dict() for p in self.poses],
            "num_poses": len(self.poses),
            "num_attempts": self.num_attempts,
            "num_conformers_generated": self.num_conformers_generated,
            "error_message": self.error_message,
            "elapsed_time": self.elapsed_time,
        }


def get_file_stem(path: Path) -> str:
    """Get clean filename stem without extension."""
    return path.stem.replace("_ligand", "").replace("_protein", "")


# ============================================================================
# RDKit Conformer Generation Functions
# ============================================================================

def generate_rdkit_conformers(
    ligand_path: Path,
    num_conformers: int = NUM_CONFORMERS,
    random_seed: int = RDKIT_RANDOM_SEED,
    prune_rms_thresh: float = RDKIT_PRUNE_RMS_THRESH,
) -> Tuple[Optional[Chem.Mol], List[int]]:
    """
    Generate multiple 3D conformers for a ligand using RDKit's distance geometry.
    
    Uses EmbedMultipleConfs with ETKDG (Experimental-Torsion basic Knowledge Distance Geometry)
    to generate diverse conformers.
    
    Args:
        ligand_path: Path to ligand file (SDF, MOL2, or PDB)
        num_conformers: Number of conformers to generate
        random_seed: Random seed for reproducibility
        prune_rms_thresh: Prune conformers with RMSD below this threshold
    
    Returns:
        Tuple of (RDKit Mol object with conformers, list of conformer IDs)
    """
    suffix = ligand_path.suffix.lower()
    
    # Load molecule based on file type
    try:
        if suffix == ".sdf":
            suppl = Chem.SDMolSupplier(str(ligand_path), removeHs=False)
            mol = next(iter(suppl), None)
        elif suffix == ".mol2":
            mol = Chem.MolFromMol2File(str(ligand_path), removeHs=False)
        elif suffix == ".pdb":
            mol = Chem.MolFromPDBFile(str(ligand_path), removeHs=False)
        else:
            print(f"  Warning: Unsupported file format {suffix}")
            return None, []
        
        if mol is None:
            print(f"  Warning: Could not load molecule from {ligand_path}")
            return None, []
        
        # Add hydrogens if not present
        mol = Chem.AddHs(mol)
        
        # Set up ETKDG parameters for conformer generation
        params = AllChem.ETKDGv3()
        params.randomSeed = random_seed
        params.numThreads = RDKIT_NUM_THREADS
        params.pruneRmsThresh = prune_rms_thresh
        params.useRandomCoords = True  # Better for difficult molecules
        
        # Generate conformers
        conf_ids = AllChem.EmbedMultipleConfs(
            mol, 
            numConfs=num_conformers,
            params=params
        )
        
        if len(conf_ids) == 0:
            # Fallback: try with random coordinates and less strict parameters
            print(f"  Retrying conformer generation with relaxed parameters...")
            params.useRandomCoords = True
            params.maxIterations = 500
            params.pruneRmsThresh = 0.1  # Less strict pruning
            conf_ids = AllChem.EmbedMultipleConfs(
                mol,
                numConfs=num_conformers,
                params=params
            )
        
        if len(conf_ids) == 0:
            print(f"  Warning: Could not generate conformers for {ligand_path.name}")
            return mol, []
        
        # Optimize conformers with MMFF force field
        results = AllChem.MMFFOptimizeMoleculeConfs(mol, numThreads=RDKIT_NUM_THREADS)
        
        # Filter out failed optimizations
        valid_conf_ids = [
            conf_id for conf_id, (converged, energy) in zip(conf_ids, results)
            if converged == 0  # 0 means optimization converged
        ]
        
        # If all failed, use original conformers
        if not valid_conf_ids:
            valid_conf_ids = list(conf_ids)
        
        return mol, valid_conf_ids
        
    except Exception as e:
        print(f"  Error generating conformers for {ligand_path.name}: {e}")
        return None, []


def save_conformer_to_sdf(mol: Chem.Mol, conf_id: int, output_path: Path) -> bool:
    """Save a specific conformer to an SDF file."""
    try:
        writer = Chem.SDWriter(str(output_path))
        writer.write(mol, confId=conf_id)
        writer.close()
        return output_path.exists() and output_path.stat().st_size > 0
    except Exception as e:
        print(f"  Error saving conformer to {output_path}: {e}")
        return False


def compute_centroid(sdf_path: Path) -> Optional[Tuple[float, float, float]]:
    """Compute the centroid of atom coordinates from an SDF file."""
    try:
        coords = []
        with open(sdf_path, 'r') as f:
            lines = f.readlines()
        
        if len(lines) < 5:
            return None
            
        counts_line = lines[3].strip()
        parts = counts_line.split()
        if len(parts) < 2:
            return None
        
        num_atoms = int(parts[0])
        
        for i in range(4, min(4 + num_atoms, len(lines))):
            parts = lines[i].split()
            if len(parts) >= 3:
                try:
                    x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
                    coords.append((x, y, z))
                except ValueError:
                    continue
        
        if not coords:
            return None
        
        coords_array = np.array(coords)
        centroid = coords_array.mean(axis=0)
        return tuple(centroid)
    except Exception as e:
        print(f"Warning: Could not compute centroid for {sdf_path}: {e}")
        return None


def compute_pose_rmsd(sdf1: Path, sdf2: Path) -> Optional[float]:
    """Compute RMSD between two poses from SDF files."""
    try:
        def read_coords(sdf_path: Path) -> Optional[np.ndarray]:
            coords = []
            with open(sdf_path, 'r') as f:
                lines = f.readlines()
            if len(lines) < 5:
                return None
            counts_line = lines[3].strip()
            parts = counts_line.split()
            if len(parts) < 2:
                return None
            num_atoms = int(parts[0])
            for i in range(4, min(4 + num_atoms, len(lines))):
                parts = lines[i].split()
                if len(parts) >= 3:
                    try:
                        x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
                        coords.append([x, y, z])
                    except ValueError:
                        continue
            return np.array(coords) if coords else None
        
        coords1 = read_coords(sdf1)
        coords2 = read_coords(sdf2)
        
        if coords1 is None or coords2 is None:
            return None
        if len(coords1) != len(coords2):
            return None
        
        diff = coords1 - coords2
        rmsd = np.sqrt((diff ** 2).sum() / len(coords1))
        return float(rmsd)
    except Exception as e:
        print(f"Warning: Could not compute RMSD: {e}")
        return None


def is_pose_unique(new_sdf: Path, existing_poses: List[PoseInfo], threshold: float) -> bool:
    """Check if a new pose is sufficiently different from existing poses."""
    if not existing_poses:
        return True
    
    for pose in existing_poses:
        rmsd = compute_pose_rmsd(new_sdf, pose.sdf_path)
        if rmsd is not None and rmsd < threshold:
            return False
    return True


def prepare_protein_with_reduce(protein_pdb: Path, output_dir: Path) -> Path:
    """Run reduce on protein to add hydrogens, or copy as-is if reduce unavailable."""
    output_pdb = output_dir / f"{protein_pdb.stem}_protein.pdb"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    if REDUCE_EXECUTABLE:
        try:
            cmd = [REDUCE_EXECUTABLE, "-Quiet", str(protein_pdb)]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
            if result.returncode == 0 and result.stdout.strip():
                output_pdb.write_text(result.stdout)
                return output_pdb
        except Exception as e:
            print(f"  Warning: reduce failed for {protein_pdb.name}: {e}")
    
    # Fallback: copy protein as-is
    shutil.copy2(protein_pdb, output_pdb)
    return output_pdb


def prepare_ligand_file(ligand_path: Path, output_dir: Path) -> Path:
    """Prepare ligand file for EquiBind (copy SDF/MOL2/PDB with proper naming)."""
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = get_file_stem(ligand_path)
    output_path = output_dir / f"{stem}_ligand{ligand_path.suffix}"
    shutil.copy2(ligand_path, output_path)
    return output_path


def collect_files(root: Path, extensions: List[str]) -> List[Path]:
    """Return files with given extensions located directly inside root (no recursion)."""
    files = []
    for ext in extensions:
        files.extend(sorted(p for p in root.glob(f"*{ext}") if p.is_file()))
    return sorted(set(files))


print("Helper functions and RDKit conformer generation defined successfully.")

Helper functions and RDKit conformer generation defined successfully.


In [5]:
def run_equibind_multiligand(
    protein_pdb: Path,
    ligand_file: Path,
    output_dir: Path,
    seed: int = 1,
    device: str = "cpu",
) -> Tuple[bool, Optional[Path], str]:
    """
    Run EquiBind multiligand_inference.py for a protein with ligand file.
    
    Uses conda environment 'equibind' and runs the command:
    conda run -n equibind python ~/docking_tools/EquiBind/multiligand_inference.py \
      -o ./equibind_out \
      -r protein.pdb \
      -l ligand.sdf \
      --device cpu
    
    Returns: (success, output_sdf_path, error_message)
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Build the command - use conda run to execute in equibind environment
    cmd = [
        "conda", "run", "-n", "equibind", "--no-capture-output",
        "python", str(EQUIBIND_MULTILIGAND_SCRIPT),
        "-o", str(output_dir),
        "-r", str(protein_pdb),
        "-l", str(ligand_file),
        "--seed", str(seed),
        "--device", device,
    ]
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=600,  # 10 minute timeout per docking
            cwd=str(output_dir.parent),
        )
        
        # Check for output.sdf in output directory
        output_sdf = output_dir / "output.sdf"
        if output_sdf.exists() and output_sdf.stat().st_size > 0:
            return True, output_sdf, ""
        
        # Check alternative locations
        for sdf in output_dir.rglob("*.sdf"):
            if sdf.stat().st_size > 0 and "failed" not in sdf.name.lower():
                return True, sdf, ""
        
        error_msg = f"No output SDF found. stdout: {result.stdout[-500:] if result.stdout else ''} stderr: {result.stderr[-500:] if result.stderr else ''}"
        return False, None, error_msg
        
    except subprocess.TimeoutExpired:
        return False, None, "Docking timeout (600s)"
    except Exception as e:
        return False, None, str(e)


print("EquiBind execution function defined (using conda environment 'equibind').")

EquiBind execution function defined (using conda environment 'equibind').


In [6]:
def setup_equibind_batch_directory(
    protein_pdb: Path,
    ligand_file: Path,
    batch_base_dir: Path,
) -> Tuple[Path, Path, Path]:
    """
    Set up batch directory structure for EquiBind docking.
    
    Creates directory structure like:
    equibind_batches/sdf/LigandName__ProteinName/
        ├── ProteinName_protein.pdb
        └── LigandName_ligand.sdf
    
    Returns: (batch_dir, prepared_protein_path, prepared_ligand_path)
    """
    protein_name = get_file_stem(protein_pdb)
    ligand_name = get_file_stem(ligand_file)
    ligand_ext = ligand_file.suffix.lower().lstrip('.')  # 'sdf', 'mol2', or 'pdb'
    
    combo_name = f"{ligand_name}__{protein_name}"
    batch_dir = batch_base_dir / ligand_ext / combo_name
    batch_dir.mkdir(parents=True, exist_ok=True)
    
    # Prepare protein file (with reduce if available)
    protein_dest = batch_dir / f"{protein_name}_protein.pdb"
    if not protein_dest.exists():
        protein_dest = prepare_protein_with_reduce(protein_pdb, batch_dir)
        expected_name = batch_dir / f"{protein_name}_protein.pdb"
        if protein_dest != expected_name and protein_dest.exists():
            shutil.move(protein_dest, expected_name)
            protein_dest = expected_name
    
    # Prepare ligand file
    ligand_dest = batch_dir / f"{ligand_name}_ligand{ligand_file.suffix}"
    if not ligand_dest.exists():
        shutil.copy2(ligand_file, ligand_dest)
    
    return batch_dir, protein_dest, ligand_dest


def dock_protein_ligand_multiple_poses(
    protein_pdb: Path,
    ligand_file: Path,
    num_poses: int = NUM_POSES,
    num_conformers: int = NUM_CONFORMERS,
    max_attempts: int = MAX_ATTEMPTS,
    rmsd_threshold: float = POSE_RMSD_THRESHOLD,
    device: str = EQUIBIND_DEVICE,
) -> DockingResult:
    """
    Dock a protein-ligand pair and generate multiple diverse poses using RDKit conformers.
    
    Strategy to generate diverse poses:
    1. Generate multiple 3D conformers using RDKit's EmbedMultipleConfs()
    2. Run EquiBind on each conformer independently
    3. Track pose RMSD to ensure diversity
    4. Save each unique pose with a distinct ID
    """
    protein_name = get_file_stem(protein_pdb)
    ligand_name = get_file_stem(ligand_file)
    combo_name = f"{ligand_name}__{protein_name}"
    
    # Set up directories
    batch_dir, prepared_protein, prepared_ligand = setup_equibind_batch_directory(
        protein_pdb, ligand_file, EQUIBIND_BATCH_DIR
    )
    # Use CONFORMER_DOCKING_OUTPUT_DIR if defined, otherwise fall back to EQUIBIND_OUTPUT_DIR
    try:
        output_dir = CONFORMER_DOCKING_OUTPUT_DIR / combo_name
    except NameError:
        output_dir = EQUIBIND_OUTPUT_DIR / combo_name
    conformers_dir = batch_dir / "conformers"
    conformers_dir.mkdir(parents=True, exist_ok=True)
    
    result = DockingResult(
        protein_name=protein_name,
        ligand_name=ligand_name,
        protein_path=protein_pdb,
        ligand_path=ligand_file,
        output_dir=output_dir,
        status="pending",
    )
    
    start_time = time.time()
    
    # Check if we should skip (existing results)
    if not OVERWRITE_EXISTING and output_dir.exists():
        existing_sdfs = list(output_dir.glob("pose_*.sdf"))
        if len(existing_sdfs) >= num_poses:
            result.status = "skipped"
            for i, sdf in enumerate(sorted(existing_sdfs)[:num_poses]):
                centroid = compute_centroid(sdf) or (0.0, 0.0, 0.0)
                result.poses.append(PoseInfo(
                    pose_id=i + 1,
                    sdf_path=sdf,
                    centroid=centroid,
                    seed=0,
                    conformer_id=0,
                ))
            result.elapsed_time = time.time() - start_time
            return result
    
    # Create/clean output directory for final poses
    output_dir.mkdir(parents=True, exist_ok=True)
    
    poses: List[PoseInfo] = []
    attempts = 0
    
    print(f"  Docking {combo_name}...")
    print(f"    Batch dir: {batch_dir}")
    
    # Step 1: Generate RDKit conformers
    print(f"    Generating {num_conformers} RDKit conformers...")
    mol, conf_ids = generate_rdkit_conformers(
        ligand_file, 
        num_conformers=num_conformers,
        random_seed=RDKIT_RANDOM_SEED,
        prune_rms_thresh=RDKIT_PRUNE_RMS_THRESH,
    )
    
    result.num_conformers_generated = len(conf_ids) if conf_ids else 0
    
    if mol is None or not conf_ids:
        print(f"    ⚠️  Could not generate RDKit conformers, falling back to original ligand")
        # Fallback: use the original ligand file
        conformer_files = [prepared_ligand]
    else:
        print(f"    ✓ Generated {len(conf_ids)} diverse conformers")
        
        # Save each conformer as a separate SDF file
        conformer_files = []
        for i, conf_id in enumerate(conf_ids):
            conf_sdf = conformers_dir / f"conformer_{i+1:02d}.sdf"
            if save_conformer_to_sdf(mol, conf_id, conf_sdf):
                conformer_files.append(conf_sdf)
        
        if not conformer_files:
            print(f"    ⚠️  Failed to save conformers, using original ligand")
            conformer_files = [prepared_ligand]
    
    # Step 2: Dock each conformer with EquiBind
    print(f"    Running EquiBind on {len(conformer_files)} conformers...")
    
    for conf_idx, conf_file in enumerate(conformer_files):
        if len(poses) >= num_poses:
            break
        
        attempts += 1
        
        # Create equibind output directory for this conformer
        equibind_out = batch_dir / f"equibind_conf_{conf_idx+1:02d}"
        if equibind_out.exists():
            shutil.rmtree(equibind_out)
        equibind_out.mkdir(parents=True, exist_ok=True)
        
        # Run EquiBind on this conformer
        success, sdf_path, error = run_equibind_multiligand(
            protein_pdb=prepared_protein,
            ligand_file=conf_file,
            output_dir=equibind_out,
            seed=RDKIT_RANDOM_SEED,  # Use consistent seed since diversity comes from conformers
            device=device,
        )
        
        if not success or sdf_path is None:
            if attempts == 1:
                print(f"    ⚠️  Conformer {conf_idx+1} docking failed: {error[:100]}")
            continue
        
        # Check if this pose is unique
        if is_pose_unique(sdf_path, poses, rmsd_threshold):
            pose_id = len(poses) + 1
            final_sdf = output_dir / f"pose_{pose_id:02d}.sdf"
            shutil.copy2(sdf_path, final_sdf)
            
            centroid = compute_centroid(final_sdf) or (0.0, 0.0, 0.0)
            pose_info = PoseInfo(
                pose_id=pose_id,
                sdf_path=final_sdf,
                centroid=centroid,
                seed=RDKIT_RANDOM_SEED,
                conformer_id=conf_idx + 1,
            )
            poses.append(pose_info)
            print(f"    ✓ Pose {pose_id}/{num_poses} from conformer {conf_idx+1}")
        else:
            print(f"    - Conformer {conf_idx+1}: Pose too similar to existing, skipping...")
        
        # Clean up attempt directory to save space
        try:
            shutil.rmtree(equibind_out)
        except:
            pass
    
    # Update result
    result.poses = poses
    result.num_attempts = attempts
    result.elapsed_time = time.time() - start_time
    
    if len(poses) >= num_poses:
        result.status = "success"
    elif len(poses) > 0:
        result.status = "partial"
        result.error_message = f"Only {len(poses)}/{num_poses} unique poses generated from {len(conformer_files)} conformers"
    else:
        result.status = "failed"
        if not result.error_message:
            result.error_message = "Could not generate any poses"
    
    # Save result metadata
    metadata_path = output_dir / "docking_metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(result.to_dict(), f, indent=2)
    
    return result


print("Multi-pose docking function with RDKit conformer generation defined.")

Multi-pose docking function with RDKit conformer generation defined.


In [7]:
def run_equibind_batch_docking(
    proteins: List[Path],
    ligands: List[Path],
    num_poses: int = NUM_POSES,
    num_conformers: int = NUM_CONFORMERS,
    max_attempts: int = MAX_ATTEMPTS,
    rmsd_threshold: float = POSE_RMSD_THRESHOLD,
    device: str = EQUIBIND_DEVICE,
) -> List[DockingResult]:
    """
    Run EquiBind docking for all protein-ligand combinations.
    Uses RDKit conformer generation for diverse poses.
    """
    total_combinations = len(proteins) * len(ligands)
    
    print("=" * 80)
    print("EquiBind Batch Docking with RDKit Conformer Generation")
    print("=" * 80)
    print(f"Proteins: {len(proteins)}")
    print(f"Ligands: {len(ligands)}")
    print(f"Total combinations: {total_combinations}")
    print(f"RDKit conformers per ligand: {num_conformers}")
    print(f"Poses per combination: {num_poses}")
    print(f"Expected total poses: {total_combinations * num_poses}")
    print(f"Device: {device}")
    print()
    
    results: List[DockingResult] = []
    
    for idx, (protein, ligand) in enumerate(itertools.product(proteins, ligands), 1):
        print(f"\n[{idx}/{total_combinations}] Processing:")
        print(f"  Protein: {protein.name}")
        print(f"  Ligand: {ligand.name}")
        
        result = dock_protein_ligand_multiple_poses(
            protein_pdb=protein,
            ligand_file=ligand,
            num_poses=num_poses,
            num_conformers=num_conformers,
            max_attempts=max_attempts,
            rmsd_threshold=rmsd_threshold,
            device=device,
        )
        
        results.append(result)
        
        status_icon = {
            "success": "✓",
            "partial": "◐",
            "failed": "✗",
            "skipped": "⊘",
        }.get(result.status, "?")
        
        print(f"  {status_icon} Status: {result.status} | "
              f"Poses: {len(result.poses)}/{num_poses} | "
              f"Conformers: {result.num_conformers_generated} | "
              f"Time: {result.elapsed_time:.1f}s")
        
        if result.error_message:
            print(f"    Error: {result.error_message[:80]}")
    
    return results


def generate_docking_summary(results: List[DockingResult]) -> Dict:
    """Generate comprehensive summary of docking results."""
    summary = {
        "timestamp": datetime.now().isoformat(),
        "configuration": {
            "num_poses_requested": NUM_POSES,
            "num_conformers": NUM_CONFORMERS,
            "max_attempts": MAX_ATTEMPTS,
            "rmsd_threshold": POSE_RMSD_THRESHOLD,
            "rdkit_prune_rms_thresh": RDKIT_PRUNE_RMS_THRESH,
            "device": EQUIBIND_DEVICE,
        },
        "overall": {
            "total_combinations": len(results),
            "successful": sum(1 for r in results if r.status == "success"),
            "partial": sum(1 for r in results if r.status == "partial"),
            "failed": sum(1 for r in results if r.status == "failed"),
            "skipped": sum(1 for r in results if r.status == "skipped"),
            "total_poses_generated": sum(len(r.poses) for r in results),
            "total_conformers_generated": sum(r.num_conformers_generated for r in results),
            "total_time_seconds": sum(r.elapsed_time for r in results),
        },
        "by_protein": defaultdict(lambda: {"combinations": 0, "poses": 0, "conformers": 0, "success": 0}),
        "by_ligand": defaultdict(lambda: {"combinations": 0, "poses": 0, "conformers": 0, "success": 0}),
        "combinations": [],
    }
    
    for r in results:
        summary["by_protein"][r.protein_name]["combinations"] += 1
        summary["by_protein"][r.protein_name]["poses"] += len(r.poses)
        summary["by_protein"][r.protein_name]["conformers"] += r.num_conformers_generated
        if r.status == "success":
            summary["by_protein"][r.protein_name]["success"] += 1
            
        summary["by_ligand"][r.ligand_name]["combinations"] += 1
        summary["by_ligand"][r.ligand_name]["poses"] += len(r.poses)
        summary["by_ligand"][r.ligand_name]["conformers"] += r.num_conformers_generated
        if r.status == "success":
            summary["by_ligand"][r.ligand_name]["success"] += 1
        
        summary["combinations"].append({
            "protein": r.protein_name,
            "ligand": r.ligand_name,
            "status": r.status,
            "num_poses": len(r.poses),
            "num_conformers": r.num_conformers_generated,
            "attempts": r.num_attempts,
            "time_seconds": round(r.elapsed_time, 2),
            "error": r.error_message if r.error_message else None,
        })
    
    summary["by_protein"] = dict(summary["by_protein"])
    summary["by_ligand"] = dict(summary["by_ligand"])
    
    return summary


def print_docking_summary(summary: Dict):
    """Print formatted docking summary."""
    print("\n" + "=" * 80)
    print("EQUIBIND DOCKING SUMMARY (RDKit Conformer Generation)")
    print("=" * 80)
    
    overall = summary["overall"]
    print(f"\nOverall Statistics:")
    print(f"  Total combinations: {overall['total_combinations']}")
    print(f"  Successful: {overall['successful']}")
    print(f"  Partial: {overall['partial']}")
    print(f"  Failed: {overall['failed']}")
    print(f"  Skipped: {overall['skipped']}")
    print(f"  Total poses generated: {overall['total_poses_generated']}")
    print(f"  Total conformers generated: {overall['total_conformers_generated']}")
    print(f"  Total time: {overall['total_time_seconds']:.1f}s "
          f"({overall['total_time_seconds']/60:.1f} min)")
    
    print("\n" + "-" * 80)
    print("Poses by Protein:")
    print("-" * 80)
    for protein, stats in sorted(summary["by_protein"].items()):
        print(f"  {protein:40s} | "
              f"Combos: {stats['combinations']:3d} | "
              f"Poses: {stats['poses']:4d} | "
              f"Confs: {stats['conformers']:4d}")
    
    print("\n" + "-" * 80)
    print("Poses by Ligand:")
    print("-" * 80)
    for ligand, stats in sorted(summary["by_ligand"].items()):
        print(f"  {ligand:40s} | "
              f"Combos: {stats['combinations']:3d} | "
              f"Poses: {stats['poses']:4d} | "
              f"Confs: {stats['conformers']:4d}")
    
    print("\n" + "-" * 80)
    print("Detailed Results per Combination:")
    print("-" * 80)
    print(f"{'Protein':<22s} {'Ligand':<22s} {'Status':<8s} {'Poses':<6s} {'Confs':<6s} {'Time':<8s}")
    print("-" * 80)
    for combo in summary["combinations"]:
        print(f"{combo['protein'][:21]:<22s} "
              f"{combo['ligand'][:21]:<22s} "
              f"{combo['status']:<8s} "
              f"{combo['num_poses']:<6d} "
              f"{combo['num_conformers']:<6d} "
              f"{combo['time_seconds']:.1f}s")


print("Batch docking and summary functions defined.")

Batch docking and summary functions defined.


In [8]:
# Collect input files
# Ligands: PDB, SDF, MOL2 files from drugs_dir
# Proteins: PDB files from receptors_dir

ligand_files = collect_files(drugs_dir, [".pdb", ".sdf", ".mol2"])
receptor_files = collect_files(receptors_dir, [".pdb"])

print("=" * 80)
print("INPUT FILES FOR EQUIBIND DOCKING")
print("=" * 80)

print(f"\nProteins ({len(receptor_files)} files from {receptors_dir}):")
for pdb in receptor_files:
    print(f"  - {pdb.name}")

print(f"\nLigands ({len(ligand_files)} files from {drugs_dir}):")
for lig in ligand_files:
    print(f"  - {lig.name}")

print(f"\nTotal docking combinations: {len(receptor_files) * len(ligand_files)}")
print(f"Poses per combination: {NUM_POSES}")
print(f"Expected total poses: {len(receptor_files) * len(ligand_files) * NUM_POSES}")

INPUT FILES FOR EQUIBIND DOCKING

Proteins (4 files from /home/manndo/MasterProject/Orai):
  - Orai1WT-MDSnap-Fr300.pdb
  - Orai1WT-MDSnap-Fr400.pdb
  - Orai1WT-MDSnap-Fr499.pdb
  - Orai1WT-START-Fr0.pdb

Ligands (5 files from /home/manndo/MasterProject/Drugs):
  - 2abp-nh2-OPT.pdb
  - 2abp-nh3p-OPT.pdb
  - Synta-66-OPT-Singlet.pdb
  - gsk7975a-deprot-OPT.pdb
  - gsk7975a-prot-OPT.pdb

Total docking combinations: 20
Poses per combination: 30
Expected total poses: 600


In [9]:
# ============================================================================
# RUN EQUIBIND BATCH DOCKING
# ============================================================================

equibind_results = run_equibind_batch_docking(
    proteins=receptor_files,
    ligands=ligand_files,
    num_poses=NUM_POSES,
    max_attempts=MAX_ATTEMPTS,
    rmsd_threshold=POSE_RMSD_THRESHOLD,
)

# Generate and display summary
docking_summary = generate_docking_summary(equibind_results)
print_docking_summary(docking_summary)

# Save summary to file
summary_path = EQUIBIND_OUTPUT_DIR / "docking_summary.json"
with open(summary_path, 'w') as f:
    json.dump(docking_summary, f, indent=2)
print(f"\nSummary saved to: {summary_path}")

EquiBind Batch Docking with RDKit Conformer Generation
Proteins: 4
Ligands: 5
Total combinations: 20
RDKit conformers per ligand: 10
Poses per combination: 30
Expected total poses: 600
Device: cpu


[1/20] Processing:
  Protein: Orai1WT-MDSnap-Fr300.pdb
  Ligand: 2abp-nh2-OPT.pdb
  Docking 2abp-nh2-OPT__Orai1WT-MDSnap-Fr300...
    Batch dir: /home/manndo/MasterProject/equibind_batches/pdb/2abp-nh2-OPT__Orai1WT-MDSnap-Fr300
    Generating 10 RDKit conformers...
    ✓ Generated 5 diverse conformers
    Running EquiBind on 5 conformers...
    ✓ Pose 1/30 from conformer 1
    ✓ Pose 2/30 from conformer 2
    ✓ Pose 3/30 from conformer 3
    ✓ Pose 4/30 from conformer 4
    ✓ Pose 5/30 from conformer 5
  ◐ Status: partial | Poses: 5/30 | Conformers: 5 | Time: 22.1s
    Error: Only 5/30 unique poses generated from 5 conformers

[2/20] Processing:
  Protein: Orai1WT-MDSnap-Fr300.pdb
  Ligand: 2abp-nh3p-OPT.pdb
  Docking 2abp-nh3p-OPT__Orai1WT-MDSnap-Fr300...
    Batch dir: /home/manndo/Master

In [10]:
# ============================================================================
# VISUALIZE RESULTS
# ============================================================================
import pandas as pd

# Convert results to DataFrame
results_data = []
for r in equibind_results:
    for pose in r.poses:
        results_data.append({
            "protein": r.protein_name,
            "ligand": r.ligand_name,
            "pose_id": pose.pose_id,
            "conformer_id": pose.conformer_id,
            "sdf_path": str(pose.sdf_path),
            "centroid_x": pose.centroid[0],
            "centroid_y": pose.centroid[1],
            "centroid_z": pose.centroid[2],
            "status": r.status,
            "num_conformers_generated": r.num_conformers_generated,
        })

results_df = pd.DataFrame(results_data)

print("=" * 80)
print("DOCKING RESULTS DATAFRAME")
print("=" * 80)
print(f"\nTotal poses in dataframe: {len(results_df)}")

if not results_df.empty:
    print("\nPoses per protein-ligand combination:")
    pose_counts = results_df.groupby(["protein", "ligand"]).agg({
        "pose_id": "count",
        "conformer_id": lambda x: list(x),
        "num_conformers_generated": "first"
    }).reset_index()
    pose_counts.columns = ["protein", "ligand", "num_poses", "conformer_ids_used", "conformers_generated"]
    display(pose_counts)
    
    print("\nPoses summary by status:")
    status_summary = results_df.groupby("status").size()
    print(status_summary)
    
    print("\nConformer distribution:")
    print(f"  Mean conformers generated per combination: {results_df.groupby(['protein', 'ligand'])['num_conformers_generated'].first().mean():.1f}")

# Save to CSV
csv_path = EQUIBIND_OUTPUT_DIR / "equibind_poses.csv"
results_df.to_csv(csv_path, index=False)
print(f"\nResults saved to: {csv_path}")

DOCKING RESULTS DATAFRAME

Total poses in dataframe: 67

Poses per protein-ligand combination:


,protein,ligand,num_poses,conformer_ids_used,conformers_generated
0,Orai1WT-MDSnap-Fr300,2abp-nh2-OPT,5,"[1, 2, 3, 4, 5]",5
1,Orai1WT-MDSnap-Fr300,2abp-nh3p-OPT,3,"[1, 2, 3]",3
2,Orai1WT-MDSnap-Fr300,Synta-66-OPT-Singlet,4,"[1, 2, 3, 4]",4
3,Orai1WT-MDSnap-Fr300,gsk7975a-deprot-OPT,2,"[1, 2]",2
4,Orai1WT-MDSnap-Fr300,gsk7975a-prot-OPT,3,"[1, 2, 3]",3
5,Orai1WT-MDSnap-Fr400,2abp-nh2-OPT,5,"[1, 2, 3, 4, 5]",5
6,Orai1WT-MDSnap-Fr400,2abp-nh3p-OPT,3,"[1, 2, 3]",3
7,Orai1WT-MDSnap-Fr400,Synta-66-OPT-Singlet,4,"[1, 2, 3, 4]",4
8,Orai1WT-MDSnap-Fr400,gsk7975a-deprot-OPT,2,"[1, 2]",2
9,Orai1WT-MDSnap-Fr400,gsk7975a-prot-OPT,3,"[1, 2, 3]",3



Poses summary by status:
status
partial    67
dtype: int64

Conformer distribution:
  Mean conformers generated per combination: 3.4

Results saved to: /home/manndo/MasterProject/equibind_docked_poses/equibind_poses.csv


In [11]:
# ============================================================================
# LIST ALL GENERATED POSES
# ============================================================================

def list_pose_files(output_dir: Path) -> Dict[str, List[Path]]:
    """List all generated pose files organized by combination."""
    poses_by_combo = {}
    
    if not output_dir.exists():
        return poses_by_combo
    
    for combo_dir in sorted(output_dir.iterdir()):
        if not combo_dir.is_dir():
            continue
        
        pose_files = sorted(combo_dir.glob("pose_*.sdf"))
        if pose_files:
            poses_by_combo[combo_dir.name] = pose_files
    
    return poses_by_combo


pose_files = list_pose_files(EQUIBIND_OUTPUT_DIR)

print("=" * 80)
print("GENERATED POSE FILES")
print("=" * 80)
print(f"\nOutput directory: {EQUIBIND_OUTPUT_DIR}")
print(f"Total combinations with poses: {len(pose_files)}")
print()

total_poses = 0
for combo_name, files in pose_files.items():
    print(f"\n{combo_name}/")
    for f in files:
        print(f"  └── {f.name}")
        total_poses += 1

print("\n" + "-" * 80)
print(f"TOTAL POSES GENERATED: {total_poses}")
print("-" * 80)

GENERATED POSE FILES

Output directory: /home/manndo/MasterProject/equibind_docked_poses
Total combinations with poses: 20


2abp-nh2-OPT__Orai1WT-MDSnap-Fr300/
  └── pose_01.sdf
  └── pose_02.sdf
  └── pose_03.sdf
  └── pose_04.sdf
  └── pose_05.sdf

2abp-nh2-OPT__Orai1WT-MDSnap-Fr400/
  └── pose_01.sdf
  └── pose_02.sdf
  └── pose_03.sdf
  └── pose_04.sdf
  └── pose_05.sdf

2abp-nh2-OPT__Orai1WT-MDSnap-Fr499/
  └── pose_01.sdf
  └── pose_02.sdf
  └── pose_03.sdf
  └── pose_04.sdf
  └── pose_05.sdf

2abp-nh2-OPT__Orai1WT-START-Fr0/
  └── pose_01.sdf
  └── pose_02.sdf
  └── pose_03.sdf
  └── pose_04.sdf
  └── pose_05.sdf

2abp-nh3p-OPT__Orai1WT-MDSnap-Fr300/
  └── pose_01.sdf
  └── pose_02.sdf
  └── pose_03.sdf

2abp-nh3p-OPT__Orai1WT-MDSnap-Fr400/
  └── pose_01.sdf
  └── pose_02.sdf
  └── pose_03.sdf

2abp-nh3p-OPT__Orai1WT-MDSnap-Fr499/
  └── pose_01.sdf
  └── pose_02.sdf
  └── pose_03.sdf

2abp-nh3p-OPT__Orai1WT-START-Fr0/
  └── pose_01.sdf
  └── pose_02.sdf
  └── pose_03.sdf

Synta

# Exclusion Approach

In [12]:
# ============================================================================
# SPATIAL EXCLUSION DOCKING CONFIGURATION
# ============================================================================
from pathlib import Path
import shutil
import time
import json
from datetime import datetime
import itertools

# EquiBind paths (if not already defined)
EQUIBIND_DIR = Path("/home/manndo/docking_tools/EquiBind")
EQUIBIND_MULTILIGAND_SCRIPT = EQUIBIND_DIR / "multiligand_inference.py"

# Device for EquiBind (cpu or cuda)
EQUIBIND_DEVICE = "cpu"

# Output directories - SEPARATE folders for each method
workspace_root = Path.cwd()
EQUIBIND_BATCH_DIR = workspace_root / "equibind_batches"

# Method 1: Standard conformer-based docking (no exclusion)
CONFORMER_DOCKING_OUTPUT_DIR = workspace_root / "equibind_conformer_poses"

# Method 2: Spatial exclusion docking
SPATIAL_EXCLUSION_OUTPUT_DIR = workspace_root / "equibind_spatial_exclusion_poses"

# Legacy output directory (for backward compatibility)
EQUIBIND_OUTPUT_DIR = workspace_root / "equibind_docked_poses"

# Create all directories
EQUIBIND_BATCH_DIR.mkdir(parents=True, exist_ok=True)
CONFORMER_DOCKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPATIAL_EXCLUSION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EQUIBIND_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# RDKit random seeds - use multiple seeds for maximum conformer diversity
# Each seed will generate a batch of conformers with different starting geometries
RDKIT_RANDOM_SEEDS: list = [42, 123, 456, 789, 1001, 2022, 3141, 5926, 8675, 9999]

# Number of conformers to generate per seed (total conformers = len(seeds) * CONFORMERS_PER_SEED)
CONFORMERS_PER_SEED: int = 6

# Number of distinct binding sites to discover per protein-ligand combination
NUM_BINDING_SITES: int = 10

# Exclusion radius (Angstroms) around each found binding site
# Poses with centroids within this distance of a previous site are rejected
EXCLUSION_RADIUS: float = 5.0  # Typical binding pocket is ~10-15Å across

# Minimum distance between binding site centroids to be considered distinct
MIN_SITE_DISTANCE: float = 3.0  # Å

# Maximum total conformers to try before giving up on finding a new site
MAX_CONFORMERS_PER_SITE: int = 90

print("=" * 80)
print("SPATIAL EXCLUSION DOCKING CONFIGURATION")
print("=" * 80)
print(f"EquiBind device: {EQUIBIND_DEVICE}")
print(f"\nOutput directories:")
print(f"  Conformer docking:     {CONFORMER_DOCKING_OUTPUT_DIR}")
print(f"  Spatial exclusion:     {SPATIAL_EXCLUSION_OUTPUT_DIR}")
print(f"\nRDKit random seeds: {RDKIT_RANDOM_SEEDS}")
print(f"Conformers per seed: {CONFORMERS_PER_SEED}")
print(f"Total conformers per batch: {len(RDKIT_RANDOM_SEEDS) * CONFORMERS_PER_SEED}")
print(f"\nSpatial exclusion settings:")
print(f"  Binding sites to discover: {NUM_BINDING_SITES}")
print(f"  Maximum conformers to try per site: {MAX_CONFORMERS_PER_SITE}")
print(f"  Exclusion radius: {EXCLUSION_RADIUS} Å")
print(f"  Minimum site distance: {MIN_SITE_DISTANCE} Å")
print("=" * 80)

SPATIAL EXCLUSION DOCKING CONFIGURATION
EquiBind device: cpu

Output directories:
  Conformer docking:     /home/manndo/MasterProject/equibind_conformer_poses
  Spatial exclusion:     /home/manndo/MasterProject/equibind_spatial_exclusion_poses

RDKit random seeds: [42, 123, 456, 789, 1001, 2022, 3141, 5926, 8675, 9999]
Conformers per seed: 6
Total conformers per batch: 60

Spatial exclusion settings:
  Binding sites to discover: 10
  Maximum conformers to try per site: 90
  Exclusion radius: 5.0 Å
  Minimum site distance: 3.0 Å


In [13]:
# ============================================================================
# SPATIAL EXCLUSION HELPER FUNCTIONS
# ============================================================================
from dataclasses import dataclass, field
from typing import Tuple, List, Optional
from pathlib import Path
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

# Configuration for poses per binding site
POSES_PER_SITE = 5  # Number of different poses to find within each binding site
SITE_INCLUSION_RADIUS = 6.0  # Radius (Å) within which poses are considered "in the same site"
MIN_POSE_RMSD = 1.5  # Minimum RMSD (Å) between poses within a site for diversity

@dataclass
class ExclusionZone:
    """Represents a spherical exclusion zone around a found binding site."""
    center: Tuple[float, float, float]  # Centroid of the binding site
    radius: float  # Exclusion radius in Angstroms
    site_id: int  # Which binding site this represents
    pose_path: Path  # Path to the pose that defined this zone
    
    def contains_point(self, point: Tuple[float, float, float]) -> bool:
        """Check if a point (x, y, z) falls within this exclusion zone."""
        dx = point[0] - self.center[0]
        dy = point[1] - self.center[1]
        dz = point[2] - self.center[2]
        distance = np.sqrt(dx**2 + dy**2 + dz**2)
        return distance < self.radius
    
    def distance_to_point(self, point: Tuple[float, float, float]) -> float:
        """Calculate distance from point to the center of exclusion zone."""
        dx = point[0] - self.center[0]
        dy = point[1] - self.center[1]
        dz = point[2] - self.center[2]
        return np.sqrt(dx**2 + dy**2 + dz**2)


@dataclass
class PoseInSite:
    """Information about a single pose within a binding site."""
    pose_id: int  # Pose number within the site (1, 2, 3, ...)
    sdf_path: Path  # Path to the SDF file for this pose
    centroid: Tuple[float, float, float]  # Centroid of this pose
    conformer_id: int  # Which conformer was used
    seed_used: int  # Random seed used for conformer generation
    distance_to_site_center: float  # Distance from pose centroid to site center
    rmsd_to_reference: Optional[float] = None  # RMSD to the reference (first) pose
    
    def to_dict(self) -> dict:
        return {
            "pose_id": self.pose_id,
            "sdf_path": str(self.sdf_path),
            "centroid": list(self.centroid),
            "conformer_id": self.conformer_id,
            "seed_used": self.seed_used,
            "distance_to_site_center": self.distance_to_site_center,
            "rmsd_to_reference": self.rmsd_to_reference,
        }


@dataclass
class BindingSiteInfo:
    """Information about a found binding site with multiple poses."""
    site_id: int
    centroid: Tuple[float, float, float]  # Site center (from first/reference pose)
    sdf_path: Path  # Path to the reference SDF file for this site
    conformer_id: int  # Conformer that found this site
    attempt_number: int
    seed_used: int = 0  # Which random seed was used
    distance_to_nearest_site: Optional[float] = None
    poses: List[PoseInSite] = field(default_factory=list)  # Multiple poses within this site
    
    @property
    def num_poses(self) -> int:
        return len(self.poses)
    
    def to_dict(self) -> dict:
        return {
            "site_id": self.site_id,
            "centroid": list(self.centroid),
            "sdf_path": str(self.sdf_path),
            "conformer_id": self.conformer_id,
            "attempt_number": self.attempt_number,
            "seed_used": self.seed_used,
            "distance_to_nearest_site": self.distance_to_nearest_site,
            "num_poses": self.num_poses,
            "poses": [p.to_dict() for p in self.poses],
        }


@dataclass
class SpatialExclusionResult:
    """Results from spatial exclusion docking with multiple poses per site."""
    protein_name: str
    ligand_name: str
    protein_path: Path
    ligand_path: Path
    output_dir: Path
    binding_sites: List[BindingSiteInfo] = field(default_factory=list)
    exclusion_zones: List[ExclusionZone] = field(default_factory=list)
    total_conformers_tried: int = 0
    total_docking_attempts: int = 0
    rejected_in_exclusion: int = 0
    poses_rejected_outside_site: int = 0  # Poses that landed outside the target site
    poses_rejected_low_rmsd: int = 0  # Poses too similar to existing poses
    elapsed_time: float = 0.0
    status: str = "pending"
    error_message: str = ""
    seeds_used: List[int] = field(default_factory=list)
    
    @property
    def total_poses(self) -> int:
        return sum(site.num_poses for site in self.binding_sites)
    
    def to_dict(self) -> dict:
        return {
            "protein_name": self.protein_name,
            "ligand_name": self.ligand_name,
            "protein_path": str(self.protein_path),
            "ligand_path": str(self.ligand_path),
            "output_dir": str(self.output_dir),
            "status": self.status,
            "binding_sites": [s.to_dict() for s in self.binding_sites],
            "num_sites_found": len(self.binding_sites),
            "total_poses": self.total_poses,
            "total_conformers_tried": self.total_conformers_tried,
            "total_docking_attempts": self.total_docking_attempts,
            "rejected_in_exclusion": self.rejected_in_exclusion,
            "poses_rejected_outside_site": self.poses_rejected_outside_site,
            "poses_rejected_low_rmsd": self.poses_rejected_low_rmsd,
            "elapsed_time": self.elapsed_time,
            "error_message": self.error_message,
            "seeds_used": self.seeds_used,
        }


def get_ligand_centroid(mol) -> Optional[Tuple[float, float, float]]:
    """Calculate the centroid of a ligand molecule."""
    try:
        conf = mol.GetConformer()
        positions = conf.GetPositions()
        centroid = positions.mean(axis=0)
        return tuple(centroid)
    except Exception as e:
        print(f"Error calculating centroid: {e}")
        return None


def is_in_any_exclusion_zone(centroid: Tuple[float, float, float], 
                              exclusion_zones: List[ExclusionZone]) -> Tuple[bool, Optional[float]]:
    """
    Check if a centroid point falls within any exclusion zone.
    
    Args:
        centroid: (x, y, z) coordinates of the ligand centroid
        exclusion_zones: List of ExclusionZone objects
        
    Returns:
        Tuple of (is_excluded, distance_to_nearest_zone_center)
    """
    if not exclusion_zones:
        return False, None
    
    min_distance = float('inf')
    is_excluded = False
    
    for zone in exclusion_zones:
        dist = zone.distance_to_point(centroid)
        if dist < min_distance:
            min_distance = dist
        if zone.contains_point(centroid):
            is_excluded = True
    
    return is_excluded, min_distance if min_distance != float('inf') else None


def generate_diverse_conformers_batch(
    ligand_file: Path, 
    seeds: List[int] = None,
    conformers_per_seed: int = None,
) -> List[Tuple[Path, int, int]]:
    """
    Generate diverse conformers using MULTIPLE random seeds for maximum diversity.
    
    Each seed produces a different set of starting geometries, so using multiple
    seeds ensures we explore more of the conformational space.
    
    Args:
        ligand_file: Path to ligand file (SDF/MOL2/PDB)
        seeds: List of random seeds to use (default: RDKIT_RANDOM_SEEDS)
        conformers_per_seed: Number of conformers per seed (default: CONFORMERS_PER_SEED)
        
    Returns:
        List of (conformer_sdf_path, conformer_id, seed_used) tuples
    """
    # Use global defaults if not provided
    if seeds is None:
        seeds = RDKIT_RANDOM_SEEDS
    if conformers_per_seed is None:
        conformers_per_seed = CONFORMERS_PER_SEED
    
    # Load molecule
    suffix = ligand_file.suffix.lower()
    try:
        if suffix == ".sdf":
            suppl = Chem.SDMolSupplier(str(ligand_file), removeHs=False)
            mol = next(iter(suppl), None)
        elif suffix == ".mol2":
            mol = Chem.MolFromMol2File(str(ligand_file), removeHs=False)
        elif suffix == ".pdb":
            mol = Chem.MolFromPDBFile(str(ligand_file), removeHs=False)
        else:
            print(f"  Warning: Unsupported file format {suffix}")
            return [(ligand_file, 0, 0)]
        
        if mol is None:
            print(f"  Warning: Could not load molecule from {ligand_file}")
            return [(ligand_file, 0, 0)]
    except Exception as e:
        print(f"  Error loading molecule: {e}")
        return [(ligand_file, 0, 0)]
    
    # Add hydrogens
    mol_h = Chem.AddHs(mol)
    
    # Create output directory for conformers
    conformer_dir = ligand_file.parent / f"{ligand_file.stem}_conformers"
    conformer_dir.mkdir(parents=True, exist_ok=True)
    
    conformer_files = []
    global_conf_id = 0
    
    print(f"  Generating conformers with {len(seeds)} different random seeds...")
    print(f"  Seeds: {seeds}")
    print(f"  Conformers per seed: {conformers_per_seed}")
    
    for seed_idx, seed in enumerate(seeds):
        # Set up ETKDG parameters for this seed
        params = AllChem.ETKDGv3()
        params.randomSeed = seed
        params.numThreads = 0  # Use all available threads
        params.useRandomCoords = True
        params.maxIterations = 500
        
        # Generate conformers with this seed
        try:
            conf_ids = AllChem.EmbedMultipleConfs(mol_h, numConfs=conformers_per_seed, params=params)
        except Exception as e:
            print(f"    Seed {seed}: Failed to generate conformers: {e}")
            continue
        
        if len(conf_ids) == 0:
            # Try fallback with relaxed parameters
            params.enforceChirality = False
            try:
                conf_ids = AllChem.EmbedMultipleConfs(mol_h, numConfs=conformers_per_seed, params=params)
            except:
                pass
        
        if len(conf_ids) == 0:
            print(f"    Seed {seed}: No conformers generated")
            continue
        
        # Minimize conformers
        for conf_id in conf_ids:
            try:
                AllChem.MMFFOptimizeMolecule(mol_h, confId=conf_id, maxIters=200)
            except:
                pass
        
        # Save each conformer
        seed_confs_saved = 0
        for local_idx, conf_id in enumerate(conf_ids):
            global_conf_id += 1
            conf_path = conformer_dir / f"seed{seed}_conf{local_idx+1:02d}.sdf"
            try:
                writer = Chem.SDWriter(str(conf_path))
                writer.write(mol_h, confId=conf_id)
                writer.close()
                if conf_path.exists() and conf_path.stat().st_size > 0:
                    conformer_files.append((conf_path, global_conf_id, seed))
                    seed_confs_saved += 1
            except Exception as e:
                print(f"    Warning: Could not save conformer {global_conf_id}: {e}")
        
        print(f"    Seed {seed}: Generated {seed_confs_saved} conformers")
    
    if not conformer_files:
        print("  Warning: No conformers generated, using original ligand")
        return [(ligand_file, 0, 0)]
    
    print(f"  ✓ Total: {len(conformer_files)} diverse conformers from {len(seeds)} seeds")
    return conformer_files


print("✓ Spatial exclusion helper classes and functions defined.")
print("  - ExclusionZone: Spherical exclusion zone")
print("  - PoseInSite: Information about a single pose within a binding site")
print("  - BindingSiteInfo: Information about found binding sites (with multiple poses)")
print("  - SpatialExclusionResult: Results container with to_dict()")
print("  - is_in_any_exclusion_zone(): Check if point is excluded")
print("  - generate_diverse_conformers_batch(): Generate conformers with MULTIPLE seeds")
print()
print("New configuration parameters:")
print(f"  POSES_PER_SITE = {POSES_PER_SITE}")
print(f"  SITE_INCLUSION_RADIUS = {SITE_INCLUSION_RADIUS} Å")
print(f"  MIN_POSE_RMSD = {MIN_POSE_RMSD} Å")

✓ Spatial exclusion helper classes and functions defined.
  - ExclusionZone: Spherical exclusion zone
  - PoseInSite: Information about a single pose within a binding site
  - BindingSiteInfo: Information about found binding sites (with multiple poses)
  - SpatialExclusionResult: Results container with to_dict()
  - is_in_any_exclusion_zone(): Check if point is excluded
  - generate_diverse_conformers_batch(): Generate conformers with MULTIPLE seeds

New configuration parameters:
  POSES_PER_SITE = 5
  SITE_INCLUSION_RADIUS = 6.0 Å
  MIN_POSE_RMSD = 1.5 Å


In [14]:
# ============================================================================
# MAIN SPATIAL EXCLUSION DOCKING FUNCTION (with Multiple Poses per Site)
# ============================================================================

def is_pose_diverse_from_existing(
    new_sdf: Path, 
    existing_poses: List[PoseInSite], 
    min_rmsd: float = MIN_POSE_RMSD
) -> Tuple[bool, Optional[float]]:
    """
    Check if a new pose is sufficiently different from existing poses in the site.
    
    Args:
        new_sdf: Path to the new pose SDF file
        existing_poses: List of existing PoseInSite objects
        min_rmsd: Minimum RMSD threshold for diversity
        
    Returns:
        Tuple of (is_diverse, min_rmsd_to_existing)
    """
    if not existing_poses:
        return True, None
    
    min_rmsd_found = float('inf')
    for pose in existing_poses:
        rmsd = compute_pose_rmsd(new_sdf, pose.sdf_path)
        if rmsd is not None and rmsd < min_rmsd_found:
            min_rmsd_found = rmsd
    
    if min_rmsd_found == float('inf'):
        return True, None
    
    return min_rmsd_found >= min_rmsd, min_rmsd_found


def load_existing_result(output_dir: Path) -> Optional[SpatialExclusionResult]:
    """
    Load an existing SpatialExclusionResult from a metadata JSON file.
    
    Args:
        output_dir: Directory containing spatial_exclusion_metadata.json
        
    Returns:
        SpatialExclusionResult if found and valid, None otherwise
    """
    metadata_path = output_dir / "spatial_exclusion_metadata.json"
    
    if not metadata_path.exists():
        return None
    
    try:
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)
        
        # Reconstruct binding sites with poses
        binding_sites = []
        for site_data in metadata.get("binding_sites", []):
            poses = []
            for pose_data in site_data.get("poses", []):
                pose = PoseInSite(
                    pose_id=pose_data["pose_id"],
                    sdf_path=Path(pose_data["sdf_path"]),
                    centroid=tuple(pose_data["centroid"]),
                    conformer_id=pose_data["conformer_id"],
                    seed_used=pose_data["seed_used"],
                    distance_to_site_center=pose_data["distance_to_site_center"],
                    rmsd_to_reference=pose_data.get("rmsd_to_reference"),
                )
                # Verify pose file exists
                if pose.sdf_path.exists():
                    poses.append(pose)
            
            if poses:  # Only add site if it has valid poses
                site = BindingSiteInfo(
                    site_id=site_data["site_id"],
                    centroid=tuple(site_data["centroid"]),
                    sdf_path=Path(site_data["sdf_path"]),
                    conformer_id=site_data["conformer_id"],
                    attempt_number=site_data["attempt_number"],
                    seed_used=site_data["seed_used"],
                    distance_to_nearest_site=site_data.get("distance_to_nearest_site"),
                    poses=poses,
                )
                binding_sites.append(site)
        
        # If no valid binding sites found, return None
        if not binding_sites:
            return None
        
        # Reconstruct result
        result = SpatialExclusionResult(
            protein_name=metadata["protein_name"],
            ligand_name=metadata["ligand_name"],
            protein_path=Path(metadata["protein_path"]),
            ligand_path=Path(metadata["ligand_path"]),
            output_dir=Path(metadata["output_dir"]),
            binding_sites=binding_sites,
            total_conformers_tried=metadata.get("total_conformers_tried", 0),
            total_docking_attempts=metadata.get("total_docking_attempts", 0),
            rejected_in_exclusion=metadata.get("rejected_in_exclusion", 0),
            poses_rejected_outside_site=metadata.get("poses_rejected_outside_site", 0),
            poses_rejected_low_rmsd=metadata.get("poses_rejected_low_rmsd", 0),
            elapsed_time=metadata.get("elapsed_time", 0.0),
            status=metadata.get("status", "success"),
            error_message=metadata.get("error_message", ""),
            seeds_used=metadata.get("seeds_used", []),
        )
        
        return result
        
    except Exception as e:
        print(f"  Warning: Could not load existing result from {metadata_path}: {e}")
        return None


def dock_with_spatial_exclusion(
    protein_pdb: Path,
    ligand_file: Path,
    num_binding_sites: int = NUM_BINDING_SITES,
    poses_per_site: int = POSES_PER_SITE,
    conformers_per_seed: int = CONFORMERS_PER_SEED,
    exclusion_radius: float = EXCLUSION_RADIUS,
    site_inclusion_radius: float = SITE_INCLUSION_RADIUS,
    min_site_distance: float = MIN_SITE_DISTANCE,
    min_pose_rmsd: float = MIN_POSE_RMSD,
    max_conformers_per_site: int = MAX_CONFORMERS_PER_SITE,
    device: str = EQUIBIND_DEVICE,
    seeds: List[int] = None,
    skip_existing: bool = True,
) -> SpatialExclusionResult:
    """
    Discover multiple distinct binding sites AND multiple poses within each site.
    
    Algorithm:
    1. Generate conformers using MULTIPLE random seeds for diversity
    2. For each binding site to find:
       a. Find the first pose that defines the site (outside all exclusion zones)
       b. Then find additional diverse poses WITHIN that site:
          - Dock more conformers
          - Accept poses that land within site_inclusion_radius of site center
          - Reject poses too similar to existing poses (RMSD < min_pose_rmsd)
       c. Create exclusion zone around the site center
       d. Move to next site
    3. Repeat until all sites found or conformers exhausted
    
    Args:
        protein_pdb: Path to protein PDB file
        ligand_file: Path to ligand file (SDF/MOL2/PDB)
        num_binding_sites: Number of distinct binding sites to discover
        poses_per_site: Number of different poses to find within each binding site
        conformers_per_seed: Number of conformers to generate per random seed
        exclusion_radius: Radius (Å) to exclude around found sites (for finding NEW sites)
        site_inclusion_radius: Radius (Å) within which poses are considered "in the same site"
        min_site_distance: Minimum distance (Å) between site centroids
        min_pose_rmsd: Minimum RMSD (Å) between poses within a site for diversity
        max_conformers_per_site: Max conformers before giving up on a site
        device: EquiBind device (cpu/cuda)
        seeds: List of random seeds for conformer generation (default: RDKIT_RANDOM_SEEDS)
        skip_existing: If True, skip docking if results already exist (default: True)
    
    Returns:
        SpatialExclusionResult with all discovered binding sites and poses
    """
    # Use global seeds if not provided
    if seeds is None:
        seeds = RDKIT_RANDOM_SEEDS
    
    protein_name = get_file_stem(protein_pdb)
    ligand_name = get_file_stem(ligand_file)
    combo_name = f"{ligand_name}__{protein_name}_spatial"
    
    # Set up directories
    batch_dir = EQUIBIND_BATCH_DIR / "spatial_exclusion" / combo_name
    output_dir = SPATIAL_EXCLUSION_OUTPUT_DIR / f"{combo_name}_sites"
    
    # Check if results already exist
    if skip_existing:
        existing_result = load_existing_result(output_dir)
        if existing_result is not None:
            print(f"\n{'='*80}")
            print(f"SKIPPING (results exist): {combo_name}")
            print(f"{'='*80}")
            print(f"  Status: {existing_result.status}")
            print(f"  Binding sites: {len(existing_result.binding_sites)}")
            print(f"  Total poses: {existing_result.total_poses}")
            print(f"  Output: {existing_result.output_dir}")
            return existing_result
    
    # Create directories
    batch_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Prepare protein
    prepared_protein = prepare_protein_with_reduce(protein_pdb, batch_dir)
    
    # Initialize result
    result = SpatialExclusionResult(
        protein_name=protein_name,
        ligand_name=ligand_name,
        protein_path=protein_pdb,
        ligand_path=ligand_file,
        output_dir=output_dir,
        seeds_used=seeds.copy(),
    )
    
    start_time = time.time()
    exclusion_zones: List[ExclusionZone] = []
    binding_sites: List[BindingSiteInfo] = []
    
    print(f"\n{'='*80}")
    print(f"SPATIAL EXCLUSION DOCKING (Multi-Pose): {combo_name}")
    print(f"{'='*80}")
    print(f"  Target binding sites: {num_binding_sites}")
    print(f"  Poses per site: {poses_per_site}")
    print(f"  Exclusion radius (between sites): {exclusion_radius} Å")
    print(f"  Site inclusion radius (for poses): {site_inclusion_radius} Å")
    print(f"  Minimum site distance: {min_site_distance} Å")
    print(f"  Minimum pose RMSD: {min_pose_rmsd} Å")
    print(f"  Random seeds: {seeds}")
    print(f"  Conformers per seed: {conformers_per_seed}")
    print()
    
    # Generate ALL conformers upfront using multiple seeds
    print(f"  Generating diverse conformers with {len(seeds)} random seeds...")
    conformer_batch = generate_diverse_conformers_batch(
        ligand_file,
        seeds=seeds,
        conformers_per_seed=conformers_per_seed,
    )
    total_conformers = len(conformer_batch)
    print(f"  Total conformers available: {total_conformers}")
    
    # Track which conformers we've used
    conformer_idx = 0
    
    # Search for each binding site
    for site_num in range(1, num_binding_sites + 1):
        print(f"\n  {'─'*60}")
        print(f"  Searching for Binding Site {site_num}/{num_binding_sites}")
        print(f"  {'─'*60}")
        
        if exclusion_zones:
            print(f"  Active exclusion zones: {len(exclusion_zones)}")
            for ez in exclusion_zones:
                print(f"    - Site {ez.site_id}: center ({ez.center[0]:.1f}, {ez.center[1]:.1f}, {ez.center[2]:.1f}), r={ez.radius}Å")
        
        site_found = False
        site_center = None
        site_poses: List[PoseInSite] = []
        conformers_tried_for_site = 0
        reference_sdf = None
        
        # PHASE 1: Find the first pose that defines this binding site
        print(f"\n  Phase 1: Finding site location...")
        
        while conformer_idx < total_conformers and not site_found:
            if conformers_tried_for_site >= max_conformers_per_site:
                print(f"  ⚠️  Max conformers ({max_conformers_per_site}) reached for site {site_num}")
                break
            
            conf_path, conf_id, seed_used = conformer_batch[conformer_idx]
            conformer_idx += 1
            conformers_tried_for_site += 1
            result.total_conformers_tried += 1
            
            # Create EquiBind output directory for this attempt
            equibind_out = batch_dir / f"site{site_num}_conf{conf_id:03d}_seed{seed_used}"
            if equibind_out.exists():
                shutil.rmtree(equibind_out)
            equibind_out.mkdir(parents=True, exist_ok=True)
            
            # Run EquiBind
            success, sdf_path, error = run_equibind_multiligand(
                protein_pdb=prepared_protein,
                ligand_file=conf_path,
                output_dir=equibind_out,
                seed=seed_used,
                device=device,
            )
            
            result.total_docking_attempts += 1
            
            if not success or sdf_path is None:
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Compute centroid of docked pose
            centroid = compute_centroid(sdf_path)
            if centroid is None:
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Check if centroid is in any exclusion zone
            is_excluded, dist_to_nearest = is_in_any_exclusion_zone(centroid, exclusion_zones)
            
            if is_excluded:
                result.rejected_in_exclusion += 1
                print(f"    Conf {conf_id} (seed {seed_used}): REJECTED (in exclusion zone, {dist_to_nearest:.1f}Å from nearest site)")
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Check minimum distance to existing sites
            if dist_to_nearest is not None and dist_to_nearest < min_site_distance:
                result.rejected_in_exclusion += 1
                print(f"    Conf {conf_id} (seed {seed_used}): REJECTED (too close: {dist_to_nearest:.1f}Å < {min_site_distance}Å)")
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # SUCCESS! Found a new binding site
            site_found = True
            site_center = centroid
            
            # Save reference pose to output directory
            site_dir = output_dir / f"site_{site_num:02d}"
            site_dir.mkdir(parents=True, exist_ok=True)
            reference_sdf = site_dir / f"pose_01.sdf"
            shutil.copy2(sdf_path, reference_sdf)
            
            # Create first pose info
            first_pose = PoseInSite(
                pose_id=1,
                sdf_path=reference_sdf,
                centroid=centroid,
                conformer_id=conf_id,
                seed_used=seed_used,
                distance_to_site_center=0.0,
                rmsd_to_reference=0.0,
            )
            site_poses.append(first_pose)
            
            print(f"    ✓ BINDING SITE {site_num} FOUND!")
            print(f"      Site center: ({centroid[0]:.2f}, {centroid[1]:.2f}, {centroid[2]:.2f})")
            print(f"      Reference pose: conformer {conf_id} (seed {seed_used})")
            
            # Clean up equibind output
            try:
                shutil.rmtree(equibind_out)
            except:
                pass
        
        if not site_found:
            print(f"  ✗ Could not find binding site {site_num}")
            if conformer_idx >= total_conformers:
                result.error_message = f"Exhausted all {total_conformers} conformers. Found {len(binding_sites)}/{num_binding_sites} sites"
            break
        
        # PHASE 2: Find additional poses within this binding site
        print(f"\n  Phase 2: Finding additional poses within site {site_num}...")
        print(f"    Target: {poses_per_site} poses, currently have: {len(site_poses)}")
        
        max_attempts_for_poses = min(max_conformers_per_site * 2, total_conformers - conformer_idx)
        attempts_for_poses = 0
        
        while len(site_poses) < poses_per_site and conformer_idx < total_conformers:
            if attempts_for_poses >= max_attempts_for_poses:
                print(f"    ⚠️  Max attempts ({max_attempts_for_poses}) reached for additional poses")
                break
            
            conf_path, conf_id, seed_used = conformer_batch[conformer_idx]
            conformer_idx += 1
            attempts_for_poses += 1
            result.total_conformers_tried += 1
            
            # Create EquiBind output directory
            equibind_out = batch_dir / f"site{site_num}_pose{len(site_poses)+1}_conf{conf_id:03d}"
            if equibind_out.exists():
                shutil.rmtree(equibind_out)
            equibind_out.mkdir(parents=True, exist_ok=True)
            
            # Run EquiBind
            success, sdf_path, error = run_equibind_multiligand(
                protein_pdb=prepared_protein,
                ligand_file=conf_path,
                output_dir=equibind_out,
                seed=seed_used,
                device=device,
            )
            
            result.total_docking_attempts += 1
            
            if not success or sdf_path is None:
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Compute centroid of docked pose
            centroid = compute_centroid(sdf_path)
            if centroid is None:
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Check if pose is WITHIN the current binding site
            dist_to_site = np.sqrt(
                (centroid[0] - site_center[0])**2 +
                (centroid[1] - site_center[1])**2 +
                (centroid[2] - site_center[2])**2
            )
            
            if dist_to_site > site_inclusion_radius:
                result.poses_rejected_outside_site += 1
                print(f"    Conf {conf_id}: REJECTED (outside site: {dist_to_site:.1f}Å > {site_inclusion_radius}Å)")
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Check if pose is diverse enough from existing poses
            is_diverse, min_rmsd_to_existing = is_pose_diverse_from_existing(
                sdf_path, site_poses, min_pose_rmsd
            )
            
            if not is_diverse:
                result.poses_rejected_low_rmsd += 1
                print(f"    Conf {conf_id}: REJECTED (too similar: RMSD {min_rmsd_to_existing:.2f}Å < {min_pose_rmsd}Å)")
                try:
                    shutil.rmtree(equibind_out)
                except:
                    pass
                continue
            
            # Calculate RMSD to reference pose
            rmsd_to_ref = compute_pose_rmsd(sdf_path, reference_sdf)
            
            # SUCCESS! Found a diverse pose within the site
            pose_num = len(site_poses) + 1
            pose_sdf = site_dir / f"pose_{pose_num:02d}.sdf"
            shutil.copy2(sdf_path, pose_sdf)
            
            new_pose = PoseInSite(
                pose_id=pose_num,
                sdf_path=pose_sdf,
                centroid=centroid,
                conformer_id=conf_id,
                seed_used=seed_used,
                distance_to_site_center=dist_to_site,
                rmsd_to_reference=rmsd_to_ref,
            )
            site_poses.append(new_pose)
            
            print(f"    ✓ Pose {pose_num}: conf {conf_id}, dist={dist_to_site:.1f}Å, RMSD={rmsd_to_ref:.2f}Å")
            
            # Clean up
            try:
                shutil.rmtree(equibind_out)
            except:
                pass
        
        # Create BindingSiteInfo with all poses
        site_info = BindingSiteInfo(
            site_id=site_num,
            centroid=site_center,
            sdf_path=reference_sdf,
            conformer_id=site_poses[0].conformer_id,
            attempt_number=conformers_tried_for_site,
            seed_used=site_poses[0].seed_used,
            distance_to_nearest_site=dist_to_nearest if 'dist_to_nearest' in dir() else None,
            poses=site_poses,
        )
        binding_sites.append(site_info)
        
        # Create exclusion zone for this site
        new_zone = ExclusionZone(
            center=site_center,
            radius=exclusion_radius,
            site_id=site_num,
            pose_path=reference_sdf,
        )
        exclusion_zones.append(new_zone)
        
        print(f"\n    Site {site_num} complete: {len(site_poses)}/{poses_per_site} poses found")
        print(f"      Exclusion zone created: radius {exclusion_radius}Å")
    
    # Finalize result
    result.binding_sites = binding_sites
    result.exclusion_zones = exclusion_zones
    result.elapsed_time = time.time() - start_time
    
    if len(binding_sites) >= num_binding_sites:
        result.status = "success"
    elif len(binding_sites) > 0:
        result.status = "partial"
    else:
        result.status = "failed"
    
    # Save metadata
    metadata = result.to_dict()
    metadata["exclusion_zones"] = [
        {
            "site_id": ez.site_id,
            "center": list(ez.center),
            "radius": ez.radius,
        }
        for ez in exclusion_zones
    ]
    metadata["conformer_generation"] = {
        "seeds_used": seeds,
        "conformers_per_seed": conformers_per_seed,
        "total_conformers_generated": total_conformers,
    }
    metadata["pose_parameters"] = {
        "poses_per_site": poses_per_site,
        "site_inclusion_radius": site_inclusion_radius,
        "min_pose_rmsd": min_pose_rmsd,
    }
    
    metadata_path = output_dir / "spatial_exclusion_metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # Print summary
    print(f"\n{'='*80}")
    print(f"SPATIAL EXCLUSION RESULTS (Multi-Pose): {combo_name}")
    print(f"{'='*80}")
    print(f"  Status: {result.status}")
    print(f"  Binding sites found: {len(binding_sites)}/{num_binding_sites}")
    print(f"  Total poses found: {result.total_poses}")
    print(f"  Seeds used for conformers: {seeds}")
    print(f"  Total conformers available: {total_conformers}")
    print(f"  Total conformers tried: {result.total_conformers_tried}")
    print(f"  Total docking attempts: {result.total_docking_attempts}")
    print(f"  Poses rejected (in exclusion zone): {result.rejected_in_exclusion}")
    print(f"  Poses rejected (outside site): {result.poses_rejected_outside_site}")
    print(f"  Poses rejected (low RMSD): {result.poses_rejected_low_rmsd}")
    print(f"  Elapsed time: {result.elapsed_time:.1f}s")
    print(f"  Output directory: {output_dir}")
    
    if binding_sites:
        print(f"\n  Binding Sites Summary:")
        for site in binding_sites:
            print(f"    Site {site.site_id}: {site.num_poses} poses")
            for pose in site.poses:
                print(f"      - Pose {pose.pose_id}: conf={pose.conformer_id}, seed={pose.seed_used}, "
                      f"dist={pose.distance_to_site_center:.1f}Å, RMSD={pose.rmsd_to_reference:.2f}Å")
    
    return result


print("✓ Main spatial exclusion docking function defined (with multi-pose support).")
print("✓ load_existing_result() function added for loading cached results.")
print("\nUsage:")
print("  result = dock_with_spatial_exclusion(")
print("      protein_pdb,")
print("      ligand_file,")
print("      num_binding_sites=3,        # Find 3 distinct sites")
print("      poses_per_site=5,           # 5 different poses per site")
print("      conformers_per_seed=5,      # 5 conformers per seed")
print("      exclusion_radius=8.0,       # 8Å exclusion around each site")
print("      site_inclusion_radius=6.0,  # Poses within 6Å of site center")
print("      min_pose_rmsd=1.5,          # Minimum RMSD between poses")
print("      seeds=[42, 123, 456, ...],  # Multiple seeds for diversity")
print("      skip_existing=True,         # Skip if results already exist")
print("  )")

✓ Main spatial exclusion docking function defined (with multi-pose support).
✓ load_existing_result() function added for loading cached results.

Usage:
  result = dock_with_spatial_exclusion(
      protein_pdb,
      ligand_file,
      num_binding_sites=3,        # Find 3 distinct sites
      poses_per_site=5,           # 5 different poses per site
      conformers_per_seed=5,      # 5 conformers per seed
      exclusion_radius=8.0,       # 8Å exclusion around each site
      site_inclusion_radius=6.0,  # Poses within 6Å of site center
      min_pose_rmsd=1.5,          # Minimum RMSD between poses
      seeds=[42, 123, 456, ...],  # Multiple seeds for diversity
      skip_existing=True,         # Skip if results already exist
  )


In [15]:
# ============================================================================
# BATCH SPATIAL EXCLUSION DOCKING (with Multi-Pose Support)
# ============================================================================

def run_spatial_exclusion_batch(
    proteins: List[Path],
    ligands: List[Path],
    num_binding_sites: int = NUM_BINDING_SITES,
    poses_per_site: int = POSES_PER_SITE,
    conformers_per_seed: int = CONFORMERS_PER_SEED,
    exclusion_radius: float = EXCLUSION_RADIUS,
    site_inclusion_radius: float = SITE_INCLUSION_RADIUS,
    min_site_distance: float = MIN_SITE_DISTANCE,
    min_pose_rmsd: float = MIN_POSE_RMSD,
    device: str = EQUIBIND_DEVICE,
    skip_existing: bool = True,
) -> List[SpatialExclusionResult]:
    """
    Run spatial exclusion docking for all protein-ligand combinations.
    
    Args:
        proteins: List of protein PDB paths
        ligands: List of ligand file paths
        num_binding_sites: Number of distinct binding sites to find per combination
        poses_per_site: Number of different poses to find within each binding site
        conformers_per_seed: Number of conformers to generate per random seed
        exclusion_radius: Radius (Å) of exclusion zones (between sites)
        site_inclusion_radius: Radius (Å) within which poses are "in the same site"
        min_site_distance: Minimum distance (Å) between sites
        min_pose_rmsd: Minimum RMSD (Å) between poses within a site
        device: EquiBind device (cpu/cuda)
        skip_existing: If True, skip combinations with existing results (default: True)
    
    Returns:
        List of SpatialExclusionResult objects
    """
    total_combinations = len(proteins) * len(ligands)
    
    print("=" * 80)
    print("SPATIAL EXCLUSION BATCH DOCKING (Multi-Pose)")
    print("=" * 80)
    print(f"Proteins: {len(proteins)}")
    print(f"Ligands: {len(ligands)}")
    print(f"Total combinations: {total_combinations}")
    print(f"Binding sites per combination: {num_binding_sites}")
    print(f"Poses per site: {poses_per_site}")
    print(f"Conformers per seed: {conformers_per_seed}")
    print(f"Exclusion radius: {exclusion_radius} Å")
    print(f"Site inclusion radius: {site_inclusion_radius} Å")
    print(f"Min pose RMSD: {min_pose_rmsd} Å")
    print(f"Skip existing: {skip_existing}")
    print(f"Expected total binding sites: {total_combinations * num_binding_sites}")
    print(f"Expected total poses: {total_combinations * num_binding_sites * poses_per_site}")
    print()
    
    results: List[SpatialExclusionResult] = []
    skipped_count = 0
    new_count = 0
    
    for idx, (protein, ligand) in enumerate(itertools.product(proteins, ligands), 1):
        print(f"\n[{idx}/{total_combinations}] Processing:")
        print(f"  Protein: {protein.name}")
        print(f"  Ligand: {ligand.name}")
        
        result = dock_with_spatial_exclusion(
            protein_pdb=protein,
            ligand_file=ligand,
            num_binding_sites=num_binding_sites,
            poses_per_site=poses_per_site,
            conformers_per_seed=conformers_per_seed,
            exclusion_radius=exclusion_radius,
            site_inclusion_radius=site_inclusion_radius,
            min_site_distance=min_site_distance,
            min_pose_rmsd=min_pose_rmsd,
            device=device,
            skip_existing=skip_existing,
        )
        
        results.append(result)
        
        # Check if this was skipped (elapsed_time would be from previous run if loaded)
        # A newly run result will have elapsed_time > 0 and was just computed
        if result.elapsed_time == 0 or (skip_existing and result.status in ["success", "partial"]):
            skipped_count += 1
        else:
            new_count += 1
        
        status_icon = {
            "success": "✓",
            "partial": "◐", 
            "failed": "✗",
        }.get(result.status, "?")
        
        print(f"\n  {status_icon} SUMMARY: {result.status.upper()}")
        print(f"    Sites found: {len(result.binding_sites)}/{num_binding_sites}")
        print(f"    Total poses: {result.total_poses}")
    
    # Print overall batch summary
    print("\n" + "=" * 80)
    print("BATCH SUMMARY")
    print("=" * 80)
    
    successful = sum(1 for r in results if r.status == "success")
    partial = sum(1 for r in results if r.status == "partial")
    failed = sum(1 for r in results if r.status == "failed")
    total_sites = sum(len(r.binding_sites) for r in results)
    total_poses = sum(r.total_poses for r in results)
    total_time = sum(r.elapsed_time for r in results)
    
    print(f"  Successful: {successful}/{total_combinations}")
    print(f"  Partial: {partial}/{total_combinations}")
    print(f"  Failed: {failed}/{total_combinations}")
    print(f"  Total binding sites found: {total_sites}")
    print(f"  Total poses found: {total_poses}")
    print(f"  Total time (new runs): {total_time:.1f}s ({total_time/60:.1f} min)")
    
    # Save batch summary
    summary_data = {
        "timestamp": datetime.now().isoformat(),
        "configuration": {
            "num_binding_sites": num_binding_sites,
            "poses_per_site": poses_per_site,
            "conformers_per_seed": conformers_per_seed,
            "exclusion_radius": exclusion_radius,
            "site_inclusion_radius": site_inclusion_radius,
            "min_site_distance": min_site_distance,
            "min_pose_rmsd": min_pose_rmsd,
            "skip_existing": skip_existing,
        },
        "results": [r.to_dict() for r in results],
    }
    
    summary_path = EQUIBIND_OUTPUT_DIR / "spatial_exclusion_batch_summary.json"
    with open(summary_path, 'w') as f:
        json.dump(summary_data, f, indent=2)
    print(f"\n  Batch summary saved to: {summary_path}")
    
    return results


print("✓ Batch spatial exclusion function defined (with multi-pose support and skip_existing).")

✓ Batch spatial exclusion function defined (with multi-pose support and skip_existing).


In [16]:
# ============================================================================
# TEST: Run Both Docking Methods on One Protein-Ligand Pair
# ============================================================================

if receptor_files and ligand_files:
    test_protein = receptor_files[0]
    test_ligand = ligand_files[0]
    
    print("=" * 80)
    print("RUNNING BOTH DOCKING METHODS FOR COMPARISON")
    print("=" * 80)
    print(f"Protein: {test_protein.name}")
    print(f"Ligand: {test_ligand.name}")
    print()
    
    # -------------------------------------------------------------------------
    # Method 1: Standard Conformer Docking (uses multiple conformers, no exclusion)
    # -------------------------------------------------------------------------
    print("\n" + "─" * 80)
    print("METHOD 1: CONFORMER DOCKING (without spatial exclusion)")
    print("─" * 80)
    
    conformer_result = dock_protein_ligand_multiple_poses(
        protein_pdb=test_protein,
        ligand_file=test_ligand,
        num_poses=NUM_POSES,
        num_conformers=NUM_CONFORMERS,
        max_attempts=MAX_ATTEMPTS,
        rmsd_threshold=POSE_RMSD_THRESHOLD,
        device=EQUIBIND_DEVICE,
    )
    
    print(f"\n  Result: {conformer_result.status}")
    print(f"  Poses generated: {len(conformer_result.poses)}")
    print(f"  Output: {conformer_result.output_dir}")
    
    if conformer_result.poses:
        print(f"\n  Conformer Docking Poses:")
        for pose in conformer_result.poses:
            print(f"    Pose {pose.pose_id}: centroid ({pose.centroid[0]:.2f}, {pose.centroid[1]:.2f}, {pose.centroid[2]:.2f})")
    
    # -------------------------------------------------------------------------
    # Method 2: Spatial Exclusion Docking (finds distinct binding sites with multiple poses)
    # -------------------------------------------------------------------------
    print("\n" + "─" * 80)
    print("METHOD 2: SPATIAL EXCLUSION DOCKING (Multi-Pose)")
    print("─" * 80)
    
    spatial_result = dock_with_spatial_exclusion(
        protein_pdb=test_protein,
        ligand_file=test_ligand,
        num_binding_sites=NUM_BINDING_SITES,
        poses_per_site=POSES_PER_SITE,
        conformers_per_seed=CONFORMERS_PER_SEED,
        exclusion_radius=EXCLUSION_RADIUS,
        site_inclusion_radius=SITE_INCLUSION_RADIUS,
        min_site_distance=MIN_SITE_DISTANCE,
        min_pose_rmsd=MIN_POSE_RMSD,
        device=EQUIBIND_DEVICE,
    )
    
    print(f"\n  Result: {spatial_result.status}")
    print(f"  Binding sites found: {len(spatial_result.binding_sites)}")
    print(f"  Total poses: {spatial_result.total_poses}")
    print(f"  Output: {spatial_result.output_dir}")
    
    if spatial_result.binding_sites:
        print(f"\n  Spatial Exclusion Binding Sites (with Multiple Poses):")
        for site in spatial_result.binding_sites:
            print(f"    Site {site.site_id}: centroid ({site.centroid[0]:.2f}, {site.centroid[1]:.2f}, {site.centroid[2]:.2f})")
            print(f"      Poses: {site.num_poses}/{POSES_PER_SITE}")
            for pose in site.poses:
                print(f"        - Pose {pose.pose_id}: seed={pose.seed_used}, conf={pose.conformer_id}, "
                      f"dist={pose.distance_to_site_center:.1f}Å, RMSD={pose.rmsd_to_reference:.2f}Å")
    
    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------
    print("\n" + "=" * 80)
    print("DOCKING COMPLETE - READY FOR COMPARISON")
    print("=" * 80)
    print(f"\nConformer docking output: {CONFORMER_DOCKING_OUTPUT_DIR}")
    print(f"Spatial exclusion output: {SPATIAL_EXCLUSION_OUTPUT_DIR}")
    print("\nRun the comparison cell below to analyze differences.")

else:
    print("No receptor or ligand files found. Please check your input directories.")

RUNNING BOTH DOCKING METHODS FOR COMPARISON
Protein: Orai1WT-MDSnap-Fr300.pdb
Ligand: 2abp-nh2-OPT.pdb


────────────────────────────────────────────────────────────────────────────────
METHOD 1: CONFORMER DOCKING (without spatial exclusion)
────────────────────────────────────────────────────────────────────────────────
  Docking 2abp-nh2-OPT__Orai1WT-MDSnap-Fr300...
    Batch dir: /home/manndo/MasterProject/equibind_batches/pdb/2abp-nh2-OPT__Orai1WT-MDSnap-Fr300
    Generating 10 RDKit conformers...
    ✓ Generated 5 diverse conformers
    Running EquiBind on 5 conformers...
    ✓ Pose 1/30 from conformer 1
    ✓ Pose 2/30 from conformer 2
    ✓ Pose 3/30 from conformer 3
    ✓ Pose 4/30 from conformer 4
    ✓ Pose 5/30 from conformer 5

  Result: partial
  Poses generated: 5
  Output: /home/manndo/MasterProject/equibind_conformer_poses/2abp-nh2-OPT__Orai1WT-MDSnap-Fr300

  Conformer Docking Poses:
    Pose 1: centroid (-10.78, 27.34, -6.58)
    Pose 2: centroid (-10.13, 28.06, -5.77

In [17]:
# ============================================================================
# RUN SPATIAL EXCLUSION DOCKING ON ALL PROTEIN-LIGAND COMBINATIONS
# ============================================================================

print("=" * 80)
print("RUNNING SPATIAL EXCLUSION DOCKING ON ALL COMBINATIONS")
print("=" * 80)
print(f"\nProteins: {len(receptor_files)}")
for p in receptor_files:
    print(f"  - {p.name}")
print(f"\nLigands: {len(ligand_files)}")
for l in ligand_files:
    print(f"  - {l.name}")
print(f"\nTotal combinations: {len(receptor_files) * len(ligand_files)}")
print(f"Binding sites per combination: {NUM_BINDING_SITES}")
print(f"Poses per site: {POSES_PER_SITE}")
print(f"Expected total poses: {len(receptor_files) * len(ligand_files) * NUM_BINDING_SITES * POSES_PER_SITE}")
print(f"\nSkip existing results: True (will load from cache if available)")
print()

# Run batch spatial exclusion docking
spatial_exclusion_results = run_spatial_exclusion_batch(
    proteins=receptor_files,
    ligands=ligand_files,
    num_binding_sites=NUM_BINDING_SITES,
    poses_per_site=POSES_PER_SITE,
    conformers_per_seed=CONFORMERS_PER_SEED,
    exclusion_radius=EXCLUSION_RADIUS,
    site_inclusion_radius=SITE_INCLUSION_RADIUS,
    min_site_distance=MIN_SITE_DISTANCE,
    min_pose_rmsd=MIN_POSE_RMSD,
    device=EQUIBIND_DEVICE,
    skip_existing=True,  # Skip combinations that already have results
)

print("\n" + "=" * 80)
print("SPATIAL EXCLUSION DOCKING COMPLETE")
print("=" * 80)

RUNNING SPATIAL EXCLUSION DOCKING ON ALL COMBINATIONS

Proteins: 4
  - Orai1WT-MDSnap-Fr300.pdb
  - Orai1WT-MDSnap-Fr400.pdb
  - Orai1WT-MDSnap-Fr499.pdb
  - Orai1WT-START-Fr0.pdb

Ligands: 5
  - 2abp-nh2-OPT.pdb
  - 2abp-nh3p-OPT.pdb
  - Synta-66-OPT-Singlet.pdb
  - gsk7975a-deprot-OPT.pdb
  - gsk7975a-prot-OPT.pdb

Total combinations: 20
Binding sites per combination: 10
Poses per site: 5
Expected total poses: 1000

Skip existing results: True (will load from cache if available)

SPATIAL EXCLUSION BATCH DOCKING (Multi-Pose)
Proteins: 4
Ligands: 5
Total combinations: 20
Binding sites per combination: 10
Poses per site: 5
Conformers per seed: 6
Exclusion radius: 5.0 Å
Site inclusion radius: 6.0 Å
Min pose RMSD: 1.5 Å
Skip existing: True
Expected total binding sites: 200
Expected total poses: 1000


[1/20] Processing:
  Protein: Orai1WT-MDSnap-Fr300.pdb
  Ligand: 2abp-nh2-OPT.pdb

SKIPPING (results exist): 2abp-nh2-OPT__Orai1WT-MDSnap-Fr300_spatial
  Status: partial
  Binding sites: 2
 

In [18]:
# ============================================================================
# VISUALIZE SPATIAL EXCLUSION RESULTS
# ============================================================================

def visualize_binding_sites(result: SpatialExclusionResult):
    """
    Create a 3D visualization of discovered binding sites and exclusion zones.
    """
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D
    
    if not result.binding_sites:
        print("No binding sites to visualize.")
        return
    
    fig = plt.figure(figsize=(12, 5))
    
    # 3D scatter plot of binding site centroids and poses
    ax1 = fig.add_subplot(121, projection='3d')
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(result.binding_sites)))
    
    for i, site in enumerate(result.binding_sites):
        # Plot site center
        x, y, z = site.centroid
        ax1.scatter([x], [y], [z], c=[colors[i]], s=300, marker='*', 
                   label=f'Site {site.site_id} center', edgecolors='black', linewidth=2)
        
        # Plot all poses within the site
        for pose in site.poses:
            px, py, pz = pose.centroid
            ax1.scatter([px], [py], [pz], c=[colors[i]], s=80, marker='o', 
                       alpha=0.6, edgecolors='black', linewidth=0.5)
        
        # Draw exclusion sphere (as wireframe)
        u = np.linspace(0, 2 * np.pi, 20)
        v = np.linspace(0, np.pi, 10)
        r = EXCLUSION_RADIUS
        xs = x + r * np.outer(np.cos(u), np.sin(v))
        ys = y + r * np.outer(np.sin(u), np.sin(v))
        zs = z + r * np.outer(np.ones(np.size(u)), np.cos(v))
        ax1.plot_wireframe(xs, ys, zs, color=colors[i], alpha=0.2, linewidth=0.5)
        
        # Draw site inclusion sphere (smaller, dashed)
        r_inc = SITE_INCLUSION_RADIUS
        xs_inc = x + r_inc * np.outer(np.cos(u), np.sin(v))
        ys_inc = y + r_inc * np.outer(np.sin(u), np.sin(v))
        zs_inc = z + r_inc * np.outer(np.ones(np.size(u)), np.cos(v))
        ax1.plot_wireframe(xs_inc, ys_inc, zs_inc, color=colors[i], alpha=0.4, linewidth=0.3, linestyle=':')
    
    ax1.set_xlabel('X (Å)')
    ax1.set_ylabel('Y (Å)')
    ax1.set_zlabel('Z (Å)')
    ax1.set_title(f'Binding Sites & Poses: {result.ligand_name} + {result.protein_name}')
    ax1.legend()
    
    # Bar chart of poses per site and RMSD diversity
    ax2 = fig.add_subplot(122)
    
    site_ids = [s.site_id for s in result.binding_sites]
    num_poses = [s.num_poses for s in result.binding_sites]
    avg_rmsd = []
    for s in result.binding_sites:
        rmsds = [p.rmsd_to_reference for p in s.poses if p.rmsd_to_reference is not None and p.rmsd_to_reference > 0]
        avg_rmsd.append(np.mean(rmsds) if rmsds else 0)
    
    x_pos = np.arange(len(site_ids))
    width = 0.35
    
    bars1 = ax2.bar(x_pos - width/2, num_poses, width, label='Poses found', color='steelblue')
    bars2 = ax2.bar(x_pos + width/2, avg_rmsd, width, label='Avg RMSD (Å)', color='coral')
    
    ax2.set_xlabel('Binding Site')
    ax2.set_ylabel('Value')
    ax2.set_title('Poses per Site & Diversity')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels([f'Site {s}' for s in site_ids])
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    # Add target line for poses
    ax2.axhline(y=POSES_PER_SITE, color='steelblue', linestyle='--', alpha=0.5, label=f'Target: {POSES_PER_SITE}')
    
    plt.tight_layout()
    
    # Save figure
    output_path = result.output_dir / "binding_sites_visualization.png"
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nVisualization saved to: {output_path}")


def create_pymol_script(result: SpatialExclusionResult) -> Path:
    """
    Generate a PyMOL script to visualize binding sites with exclusion spheres.
    """
    if not result.binding_sites:
        print("No binding sites to visualize.")
        return None
    
    script_path = result.output_dir / "visualize_sites.pml"
    
    colors = ['red', 'green', 'blue', 'yellow', 'magenta', 'cyan', 'orange', 'purple']
    
    script_lines = [
        f"# PyMOL visualization script for spatial exclusion docking",
        f"# Generated: {datetime.now().isoformat()}",
        f"# Protein: {result.protein_name}",
        f"# Ligand: {result.ligand_name}",
        "",
        f"# Load protein",
        f'load {result.protein_path}, protein',
        f'show cartoon, protein',
        f'color gray80, protein',
        "",
    ]
    
    for site in result.binding_sites:
        color = colors[(site.site_id - 1) % len(colors)]
        site_name = f"site_{site.site_id}"
        
        script_lines.extend([
            f"# Binding Site {site.site_id} ({site.num_poses} poses)",
        ])
        
        # Load all poses for this site
        for pose in site.poses:
            pose_name = f"site{site.site_id}_pose{pose.pose_id}"
            script_lines.extend([
                f'load {pose.sdf_path}, {pose_name}',
                f'show sticks, {pose_name}',
                f'color {color}, {pose_name}',
                f'set stick_radius, 0.15, {pose_name}',
            ])
        
        script_lines.extend([
            "",
            f"# Exclusion sphere for site {site.site_id}",
            f'pseudoatom sphere_{site.site_id}, pos=[{site.centroid[0]:.2f}, {site.centroid[1]:.2f}, {site.centroid[2]:.2f}]',
            f'show sphere, sphere_{site.site_id}',
            f'set sphere_scale, {EXCLUSION_RADIUS}, sphere_{site.site_id}',
            f'set sphere_transparency, 0.8, sphere_{site.site_id}',
            f'color {color}, sphere_{site.site_id}',
            "",
            f"# Site inclusion sphere for site {site.site_id}",
            f'pseudoatom inc_sphere_{site.site_id}, pos=[{site.centroid[0]:.2f}, {site.centroid[1]:.2f}, {site.centroid[2]:.2f}]',
            f'show sphere, inc_sphere_{site.site_id}',
            f'set sphere_scale, {SITE_INCLUSION_RADIUS}, inc_sphere_{site.site_id}',
            f'set sphere_transparency, 0.9, inc_sphere_{site.site_id}',
            f'color white, inc_sphere_{site.site_id}',
            "",
        ])
    
    script_lines.extend([
        "# Final setup",
        "bg_color white",
        "set ray_shadows, 0",
        "orient",
        "zoom all, 2",
    ])
    
    with open(script_path, 'w') as f:
        f.write('\n'.join(script_lines))
    
    print(f"PyMOL script saved to: {script_path}")
    print(f"Run in PyMOL with: @{script_path}")
    
    return script_path


print("✓ Visualization functions defined.")
print("  - visualize_binding_sites(): 3D matplotlib plot")
print("  - create_pymol_script(): PyMOL visualization script")

✓ Visualization functions defined.
  - visualize_binding_sites(): 3D matplotlib plot
  - create_pymol_script(): PyMOL visualization script


In [19]:
# ============================================================================
# COMPARE POSES FROM CONFORMER DOCKING VS SPATIAL EXCLUSION DOCKING
# ============================================================================
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass

@dataclass
class PoseComparisonResult:
    """Result of comparing poses from two methods."""
    protein_name: str
    ligand_name: str
    method1_name: str
    method2_name: str
    method1_poses: List[Path]
    method2_poses: List[Path]
    rmsd_matrix: np.ndarray  # RMSD between all pose pairs
    centroid_delta_matrix: np.ndarray  # Centroid distance between all pose pairs
    method1_centroids: List[Tuple[float, float, float]]
    method2_centroids: List[Tuple[float, float, float]]


def compute_centroid_from_sdf(sdf_path: Path) -> Optional[Tuple[float, float, float]]:
    """Compute the centroid of atom coordinates from an SDF file."""
    try:
        coords = []
        with open(sdf_path, 'r') as f:
            lines = f.readlines()
        
        if len(lines) < 5:
            return None
            
        counts_line = lines[3].strip()
        parts = counts_line.split()
        if len(parts) < 2:
            return None
        
        num_atoms = int(parts[0])
        
        for i in range(4, min(4 + num_atoms, len(lines))):
            parts = lines[i].split()
            if len(parts) >= 3:
                try:
                    x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
                    coords.append((x, y, z))
                except ValueError:
                    continue
        
        if not coords:
            return None
        
        coords_array = np.array(coords)
        centroid = coords_array.mean(axis=0)
        return tuple(centroid)
    except Exception as e:
        return None


def compute_rmsd_from_sdfs(sdf1: Path, sdf2: Path) -> Optional[float]:
    """Compute RMSD between two poses from SDF files."""
    try:
        def read_coords(sdf_path: Path) -> Optional[np.ndarray]:
            coords = []
            with open(sdf_path, 'r') as f:
                lines = f.readlines()
            if len(lines) < 5:
                return None
            counts_line = lines[3].strip()
            parts = counts_line.split()
            if len(parts) < 2:
                return None
            num_atoms = int(parts[0])
            for i in range(4, min(4 + num_atoms, len(lines))):
                parts = lines[i].split()
                if len(parts) >= 3:
                    try:
                        x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
                        coords.append([x, y, z])
                    except ValueError:
                        continue
            return np.array(coords) if coords else None
        
        coords1 = read_coords(sdf1)
        coords2 = read_coords(sdf2)
        
        if coords1 is None or coords2 is None:
            return None
        if len(coords1) != len(coords2):
            return None
        
        diff = coords1 - coords2
        rmsd = np.sqrt((diff ** 2).sum() / len(coords1))
        return float(rmsd)
    except Exception as e:
        return None


def compute_centroid_distance(c1: Tuple[float, float, float], 
                               c2: Tuple[float, float, float]) -> float:
    """Compute Euclidean distance between two centroids."""
    return np.sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2 + (c1[2]-c2[2])**2)


def collect_poses_from_directory(base_dir: Path, pattern: str = "*.sdf") -> Dict[str, List[Path]]:
    """
    Collect all poses organized by protein-ligand combination.
    
    Returns:
        Dict mapping combo_name -> list of SDF paths
    """
    poses_by_combo = {}
    
    if not base_dir.exists():
        print(f"  Warning: Directory does not exist: {base_dir}")
        return poses_by_combo
    
    # Check for subdirectories (each is a protein-ligand combo)
    for combo_dir in sorted(base_dir.iterdir()):
        if combo_dir.is_dir():
            sdf_files = sorted(combo_dir.glob(pattern))
            if sdf_files:
                poses_by_combo[combo_dir.name] = sdf_files
    
    return poses_by_combo


def compare_docking_methods(
    conformer_dir: Path,
    spatial_exclusion_dir: Path,
) -> Tuple[List[PoseComparisonResult], pd.DataFrame]:
    """
    Compare poses from conformer docking vs spatial exclusion docking.
    
    Args:
        conformer_dir: Directory containing conformer docking results
        spatial_exclusion_dir: Directory containing spatial exclusion results
        
    Returns:
        List of PoseComparisonResult objects and summary DataFrame
    """
    print("=" * 80)
    print("COMPARING DOCKING METHODS")
    print("=" * 80)
    print(f"Method 1 (Conformer docking): {conformer_dir}")
    print(f"Method 2 (Spatial exclusion): {spatial_exclusion_dir}")
    print()
    
    # Collect poses from both methods
    print("Collecting poses...")
    conformer_poses = collect_poses_from_directory(conformer_dir, "pose_*.sdf")
    spatial_poses = collect_poses_from_directory(spatial_exclusion_dir, "binding_site_*.sdf")
    
    print(f"  Conformer docking: {len(conformer_poses)} combinations")
    print(f"  Spatial exclusion: {len(spatial_poses)} combinations")
    
    # Find common protein-ligand combinations
    # Normalize names: spatial exclusion adds "_spatial_sites" suffix
    def normalize_name(name: str) -> str:
        return name.replace("_spatial_sites", "").replace("_sites", "")
    
    conformer_combos = {normalize_name(k): k for k in conformer_poses.keys()}
    spatial_combos = {normalize_name(k): k for k in spatial_poses.keys()}
    
    common_combos = set(conformer_combos.keys()) & set(spatial_combos.keys())
    print(f"\n  Common combinations: {len(common_combos)}")
    
    results = []
    summary_data = []
    
    for combo in sorted(common_combos):
        print(f"\n  Comparing: {combo}")
        
        conf_key = conformer_combos[combo]
        spat_key = spatial_combos[combo]
        
        conf_poses_list = conformer_poses[conf_key]
        spat_poses_list = spatial_poses[spat_key]
        
        print(f"    Conformer poses: {len(conf_poses_list)}")
        print(f"    Spatial exclusion poses: {len(spat_poses_list)}")
        
        # Parse protein and ligand names from combo
        parts = combo.split("__")
        if len(parts) >= 2:
            ligand_name = parts[0]
            protein_name = parts[1]
        else:
            ligand_name = combo
            protein_name = "unknown"
        
        # Compute centroids for all poses
        conf_centroids = [compute_centroid_from_sdf(p) for p in conf_poses_list]
        spat_centroids = [compute_centroid_from_sdf(p) for p in spat_poses_list]
        
        # Filter out None centroids
        valid_conf = [(p, c) for p, c in zip(conf_poses_list, conf_centroids) if c is not None]
        valid_spat = [(p, c) for p, c in zip(spat_poses_list, spat_centroids) if c is not None]
        
        if not valid_conf or not valid_spat:
            print(f"    Warning: No valid poses to compare")
            continue
        
        conf_poses_valid, conf_centroids_valid = zip(*valid_conf)
        spat_poses_valid, spat_centroids_valid = zip(*valid_spat)
        
        # Compute RMSD matrix (conformer poses x spatial poses)
        n_conf = len(conf_poses_valid)
        n_spat = len(spat_poses_valid)
        rmsd_matrix = np.full((n_conf, n_spat), np.nan)
        centroid_delta_matrix = np.full((n_conf, n_spat), np.nan)
        
        for i, conf_pose in enumerate(conf_poses_valid):
            for j, spat_pose in enumerate(spat_poses_valid):
                rmsd = compute_rmsd_from_sdfs(conf_pose, spat_pose)
                if rmsd is not None:
                    rmsd_matrix[i, j] = rmsd
                
                centroid_delta = compute_centroid_distance(
                    conf_centroids_valid[i], 
                    spat_centroids_valid[j]
                )
                centroid_delta_matrix[i, j] = centroid_delta
        
        # Create comparison result
        result = PoseComparisonResult(
            protein_name=protein_name,
            ligand_name=ligand_name,
            method1_name="Conformer",
            method2_name="Spatial Exclusion",
            method1_poses=list(conf_poses_valid),
            method2_poses=list(spat_poses_valid),
            rmsd_matrix=rmsd_matrix,
            centroid_delta_matrix=centroid_delta_matrix,
            method1_centroids=list(conf_centroids_valid),
            method2_centroids=list(spat_centroids_valid),
        )
        results.append(result)
        
        # Compute summary statistics
        valid_rmsd = rmsd_matrix[~np.isnan(rmsd_matrix)]
        valid_centroid = centroid_delta_matrix[~np.isnan(centroid_delta_matrix)]
        
        # Find best matches (minimum RMSD/centroid for each pose)
        min_rmsd_per_conf = np.nanmin(rmsd_matrix, axis=1) if rmsd_matrix.size > 0 else []
        min_rmsd_per_spat = np.nanmin(rmsd_matrix, axis=0) if rmsd_matrix.size > 0 else []
        min_centroid_per_conf = np.nanmin(centroid_delta_matrix, axis=1)
        min_centroid_per_spat = np.nanmin(centroid_delta_matrix, axis=0)
        
        summary_data.append({
            "protein": protein_name,
            "ligand": ligand_name,
            "combo": combo,
            "n_conformer_poses": n_conf,
            "n_spatial_poses": n_spat,
            "rmsd_min": np.nanmin(valid_rmsd) if len(valid_rmsd) > 0 else np.nan,
            "rmsd_max": np.nanmax(valid_rmsd) if len(valid_rmsd) > 0 else np.nan,
            "rmsd_mean": np.nanmean(valid_rmsd) if len(valid_rmsd) > 0 else np.nan,
            "rmsd_std": np.nanstd(valid_rmsd) if len(valid_rmsd) > 0 else np.nan,
            "centroid_delta_min": np.nanmin(valid_centroid) if len(valid_centroid) > 0 else np.nan,
            "centroid_delta_max": np.nanmax(valid_centroid) if len(valid_centroid) > 0 else np.nan,
            "centroid_delta_mean": np.nanmean(valid_centroid) if len(valid_centroid) > 0 else np.nan,
            "centroid_delta_std": np.nanstd(valid_centroid) if len(valid_centroid) > 0 else np.nan,
            "best_match_rmsd": np.nanmin(min_rmsd_per_conf) if len(min_rmsd_per_conf) > 0 else np.nan,
            "best_match_centroid": np.nanmin(min_centroid_per_conf) if len(min_centroid_per_conf) > 0 else np.nan,
        })
        
        print(f"    RMSD range: {np.nanmin(valid_rmsd):.2f} - {np.nanmax(valid_rmsd):.2f} Å (mean: {np.nanmean(valid_rmsd):.2f} Å)")
        print(f"    Centroid Δ range: {np.nanmin(valid_centroid):.2f} - {np.nanmax(valid_centroid):.2f} Å (mean: {np.nanmean(valid_centroid):.2f} Å)")
    
    summary_df = pd.DataFrame(summary_data)
    
    return results, summary_df


def print_detailed_comparison(results: List[PoseComparisonResult]):
    """Print detailed pose-by-pose comparison."""
    print("\n" + "=" * 100)
    print("DETAILED POSE-BY-POSE COMPARISON")
    print("=" * 100)
    
    for result in results:
        print(f"\n{'─'*80}")
        print(f"Protein: {result.protein_name} | Ligand: {result.ligand_name}")
        print(f"{'─'*80}")
        
        # Print centroid positions for Method 1
        print(f"\n  {result.method1_name} Poses ({len(result.method1_poses)}):")
        for i, (pose, centroid) in enumerate(zip(result.method1_poses, result.method1_centroids)):
            print(f"    Pose {i+1}: {pose.name}")
            print(f"      Centroid: ({centroid[0]:.2f}, {centroid[1]:.2f}, {centroid[2]:.2f})")
        
        # Print centroid positions for Method 2
        print(f"\n  {result.method2_name} Poses ({len(result.method2_poses)}):")
        for i, (pose, centroid) in enumerate(zip(result.method2_poses, result.method2_centroids)):
            print(f"    Pose {i+1}: {pose.name}")
            print(f"      Centroid: ({centroid[0]:.2f}, {centroid[1]:.2f}, {centroid[2]:.2f})")
        
        # Print RMSD matrix
        print(f"\n  RMSD Matrix ({result.method1_name} vs {result.method2_name}):")
        print(f"  {'':12s}", end="")
        for j in range(len(result.method2_poses)):
            print(f"  Spat_{j+1:02d}", end="")
        print()
        
        for i in range(len(result.method1_poses)):
            print(f"  Conf_{i+1:02d}    ", end="")
            for j in range(len(result.method2_poses)):
                val = result.rmsd_matrix[i, j]
                if np.isnan(val):
                    print(f"  {'N/A':>6s}", end="")
                else:
                    print(f"  {val:6.2f}", end="")
            print()
        
        # Print Centroid Delta matrix
        print(f"\n  Centroid Distance Matrix (Å):")
        print(f"  {'':12s}", end="")
        for j in range(len(result.method2_poses)):
            print(f"  Spat_{j+1:02d}", end="")
        print()
        
        for i in range(len(result.method1_poses)):
            print(f"  Conf_{i+1:02d}    ", end="")
            for j in range(len(result.method2_poses)):
                val = result.centroid_delta_matrix[i, j]
                print(f"  {val:6.2f}", end="")
            print()


def visualize_comparison(results: List[PoseComparisonResult], summary_df: pd.DataFrame):
    """Create visualization of the comparison results."""
    try:
        import matplotlib.pyplot as plt
    except ImportError as e:
        print(f"\n⚠️  Matplotlib not available in this environment: {e}")
        print("   Skipping visualization. The comparison results are available in the summary table above.")
        print("   To enable visualization, install matplotlib in a different environment.")
        return
    
    if summary_df.empty:
        print("No data to visualize.")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: RMSD distribution per combination
    ax1 = axes[0, 0]
    combos = summary_df['combo'].values
    x = np.arange(len(combos))
    width = 0.35
    
    ax1.bar(x - width/2, summary_df['rmsd_min'], width, label='Min RMSD', color='green', alpha=0.7)
    ax1.bar(x + width/2, summary_df['rmsd_mean'], width, label='Mean RMSD', color='blue', alpha=0.7)
    ax1.errorbar(x + width/2, summary_df['rmsd_mean'], yerr=summary_df['rmsd_std'], 
                 fmt='none', color='black', capsize=3)
    ax1.set_xlabel('Protein-Ligand Combination')
    ax1.set_ylabel('RMSD (Å)')
    ax1.set_title('RMSD: Conformer vs Spatial Exclusion')
    ax1.set_xticks(x)
    ax1.set_xticklabels([c[:20] + '...' if len(c) > 20 else c for c in combos], 
                        rotation=45, ha='right', fontsize=8)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Plot 2: Centroid distance distribution per combination
    ax2 = axes[0, 1]
    ax2.bar(x - width/2, summary_df['centroid_delta_min'], width, label='Min Δ', color='orange', alpha=0.7)
    ax2.bar(x + width/2, summary_df['centroid_delta_mean'], width, label='Mean Δ', color='red', alpha=0.7)
    ax2.errorbar(x + width/2, summary_df['centroid_delta_mean'], yerr=summary_df['centroid_delta_std'], 
                 fmt='none', color='black', capsize=3)
    ax2.set_xlabel('Protein-Ligand Combination')
    ax2.set_ylabel('Centroid Distance (Å)')
    ax2.set_title('Centroid Distance: Conformer vs Spatial Exclusion')
    ax2.set_xticks(x)
    ax2.set_xticklabels([c[:20] + '...' if len(c) > 20 else c for c in combos], 
                        rotation=45, ha='right', fontsize=8)
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    # Plot 3: Number of poses per method
    ax3 = axes[1, 0]
    ax3.bar(x - width/2, summary_df['n_conformer_poses'], width, label='Conformer', color='steelblue')
    ax3.bar(x + width/2, summary_df['n_spatial_poses'], width, label='Spatial Exclusion', color='coral')
    ax3.set_xlabel('Protein-Ligand Combination')
    ax3.set_ylabel('Number of Poses')
    ax3.set_title('Poses Generated per Method')
    ax3.set_xticks(x)
    ax3.set_xticklabels([c[:20] + '...' if len(c) > 20 else c for c in combos], 
                        rotation=45, ha='right', fontsize=8)
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)
    
    # Plot 4: Scatter of RMSD vs Centroid Distance
    ax4 = axes[1, 1]
    scatter = ax4.scatter(summary_df['rmsd_mean'], summary_df['centroid_delta_mean'], 
                          s=100, c=np.arange(len(summary_df)), cmap='viridis', 
                          edgecolors='black', linewidth=1)
    
    # Add labels
    for i, combo in enumerate(combos):
        ax4.annotate(combo[:15], (summary_df['rmsd_mean'].iloc[i], summary_df['centroid_delta_mean'].iloc[i]),
                    fontsize=7, ha='left', va='bottom')
    
    ax4.set_xlabel('Mean RMSD (Å)')
    ax4.set_ylabel('Mean Centroid Distance (Å)')
    ax4.set_title('RMSD vs Centroid Distance')
    ax4.grid(alpha=0.3)
    
    # Add diagonal reference line
    max_val = max(ax4.get_xlim()[1], ax4.get_ylim()[1])
    ax4.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='y=x')
    
    plt.tight_layout()
    
    # Save figure
    output_path = workspace_root / "pose_comparison_methods.png"
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nVisualization saved to: {output_path}")


print("✓ Pose comparison functions defined.")
print("  - compare_docking_methods(): Compare conformer vs spatial exclusion poses")
print("  - print_detailed_comparison(): Show pose-by-pose comparison")
print("  - visualize_comparison(): Create comparison plots")

✓ Pose comparison functions defined.
  - compare_docking_methods(): Compare conformer vs spatial exclusion poses
  - print_detailed_comparison(): Show pose-by-pose comparison
  - visualize_comparison(): Create comparison plots


In [20]:
# ============================================================================
# RUN COMPARISON: CONFORMER DOCKING VS SPATIAL EXCLUSION DOCKING
# ============================================================================

# Compare the poses from both methods
comparison_results, comparison_summary = compare_docking_methods(
    conformer_dir=CONFORMER_DOCKING_OUTPUT_DIR,
    spatial_exclusion_dir=SPATIAL_EXCLUSION_OUTPUT_DIR,
)

# Display summary table
print("\n" + "=" * 100)
print("COMPARISON SUMMARY TABLE")
print("=" * 100)

if not comparison_summary.empty:
    # Format the summary for display
    display_cols = [
        'combo', 'n_conformer_poses', 'n_spatial_poses',
        'rmsd_min', 'rmsd_mean', 'rmsd_max',
        'centroid_delta_min', 'centroid_delta_mean', 'centroid_delta_max',
    ]
    
    display_df = comparison_summary[display_cols].copy()
    display_df.columns = [
        'Combination', 'Conf. Poses', 'Spat. Poses',
        'RMSD Min (Å)', 'RMSD Mean (Å)', 'RMSD Max (Å)',
        'Centroid Δ Min (Å)', 'Centroid Δ Mean (Å)', 'Centroid Δ Max (Å)',
    ]
    
    # Round numerical columns
    numeric_cols = display_df.select_dtypes(include=[np.number]).columns
    display_df[numeric_cols] = display_df[numeric_cols].round(2)
    
    display(display_df)
    
    # Save to CSV
    csv_path = workspace_root / "pose_comparison_summary.csv"
    comparison_summary.to_csv(csv_path, index=False)
    print(f"\nSummary saved to: {csv_path}")
    
    # Print detailed comparison
    print_detailed_comparison(comparison_results)
    
    # Visualize
    visualize_comparison(comparison_results, comparison_summary)
else:
    print("No poses found to compare.")
    print(f"  Check that both directories contain poses:")
    print(f"    - Conformer docking: {CONFORMER_DOCKING_OUTPUT_DIR}")
    print(f"    - Spatial exclusion: {SPATIAL_EXCLUSION_OUTPUT_DIR}")

COMPARING DOCKING METHODS
Method 1 (Conformer docking): /home/manndo/MasterProject/equibind_conformer_poses
Method 2 (Spatial exclusion): /home/manndo/MasterProject/equibind_spatial_exclusion_poses

  Conformer docking: 1 combinations
  Spatial exclusion: 0 combinations

  Common combinations: 0

COMPARISON SUMMARY TABLE
No poses found to compare.
  Check that both directories contain poses:
    - Conformer docking: /home/manndo/MasterProject/equibind_conformer_poses
    - Spatial exclusion: /home/manndo/MasterProject/equibind_spatial_exclusion_poses
